# Exploracion y Transformación de Datos

Este prototipo integra las transformaciones de:

- Data Cleaning
- Feature Engineering

El objetivo es construir un flujo reproducible que pueda aplicarse a cualquier dataset de entrada.

In [21]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

import joblib

import ipywidgets as widgets
from IPython.display import display

# 📂 Carga de datos

In [22]:
# ============================================================
# 📂 CARGA DE DATOS (VERSIÓN ROBUSTA - PIPELINE)
# ============================================================

import io
import os
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

# ============================================================
# 🧩 WIDGETS
# ============================================================

uploader = widgets.FileUpload(accept='.csv', multiple=False)
btn_upload = widgets.Button(description="📂 Cargar archivo", button_style='success')

ruta = widgets.Text(
    description="Ruta CSV:",
    placeholder="Ej: data/raw/archivo.csv"
)
btn_ruta = widgets.Button(description="Cargar desde ruta", button_style='info')

output_loader = widgets.Output()

# ============================================================
# 🔧 FUNCIÓN ROBUSTA DE LECTURA
# ============================================================

def leer_csv_seguro(path_or_buffer):
    try:
        return pd.read_csv(path_or_buffer)
    except:
        try:
            return pd.read_csv(path_or_buffer, sep=';')
        except:
            return pd.read_csv(path_or_buffer, encoding='latin1')

# ============================================================
# 📤 CARGA DESDE UPLOAD
# ============================================================

def cargar_upload(b):
    global df
    with output_loader:
        output_loader.clear_output()

        if not uploader.value:
            print("⚠️ Sube un archivo primero")
            return

        try:
            archivo = next(iter(uploader.value.values()))
            contenido = archivo['content']
            df = leer_csv_seguro(io.BytesIO(contenido))

            print("✅ Dataset cargado desde upload")
            print("📐 Dimensión:", df.shape)
            display(df.head())

        except Exception as e:
            print("❌ Error al cargar archivo:")
            print(e)

# ============================================================
# 📁 CARGA DESDE RUTA
# ============================================================

def cargar_ruta(b):
    global df
    with output_loader:
        output_loader.clear_output()

        ruta_input = ruta.value.strip()

        if not ruta_input:
            print("⚠️ Ingresa una ruta")
            return

        # Validar existencia
        if not os.path.exists(ruta_input):
            print("❌ La ruta no existe")
            print("👉 Verifica:", ruta_input)
            return

        try:
            df = leer_csv_seguro(ruta_input)

            print("✅ Dataset cargado desde ruta")
            print("📐 Dimensión:", df.shape)
            display(df.head())

        except Exception as e:
            print("❌ Error al leer el CSV:")
            print(e)

# ============================================================
# 🔗 EVENTOS
# ============================================================

btn_upload.on_click(cargar_upload)
btn_ruta.on_click(cargar_ruta)

# ============================================================
# 🖥️ UI FINAL
# ============================================================

display(widgets.VBox([
    widgets.HTML("<h3>📂 Carga de datos (Pipeline)</h3>"),

    widgets.HTML("<b>📤 Subir archivo</b>"),
    uploader,
    btn_upload,

    widgets.HTML("<b>📁 O cargar desde ruta</b>"),
    ruta,
    btn_ruta,

    output_loader
]))

# Paso 1: Exploración de Datos Interactiva (EDA)

Este prototipo permite explorar un dataset de forma interactiva sin necesidad de programar.

## Objetivo
Facilitar que un stakeholder no técnico pueda:
- Visualizar variables
- Analizar relaciones con el target
- Evaluar calidad de datos
- Identificar patrones clave

## Controles disponibles
- Dropdown → seleccionar variables
- Slider → filtrar valores numéricos
- Checkbox → activar/desactivar filtros
- RadioButtons → elegir tipo de gráfico

In [24]:
# ============================================================
# EDA INTERACTIVO PRO FINAL (POKA-YOKE + TABS + TEMPORAL PRO)
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display

sns.set_theme(style="whitegrid")

# ============================================================
# 🔧 VALIDACIÓN
# ============================================================

def df_valido():
    return 'df' in globals() and isinstance(df, pd.DataFrame) and not df.empty

# ============================================================
# OUTPUTS
# ============================================================

out_uni = widgets.Output()
out_bi = widgets.Output()
out_multi = widgets.Output()
out_temp = widgets.Output()
out_status = widgets.Output()

# ============================================================
# WIDGETS
# ============================================================

btn_load = widgets.Button(description="⚡ Cargar variables", button_style='warning')

# UNIVARIADO
var_uni = widgets.Dropdown(description="Variable:")
plot_uni = widgets.RadioButtons(options=['Histograma', 'Barplot'])
btn_uni = widgets.Button(description="Analizar", button_style='success')

# BIVARIADO
var_bi_x = widgets.Dropdown(description="X:")
var_bi_y = widgets.Dropdown(description="Y:")
plot_bi = widgets.RadioButtons(options=['Scatter', 'Boxplot', 'Categórica vs Target'])
btn_bi = widgets.Button(description="Analizar", button_style='success')

# MULTIVARIADO
btn_multi = widgets.Button(description="Heatmap", button_style='success')

# TEMPORAL
var_temp_target = widgets.Dropdown(description="Target:")
time_granularity = widgets.RadioButtons(
    options=['Año', 'Mes', 'Semana', 'Día'],
    description='Nivel:'
)
agg_type = widgets.RadioButtons(
    options=['Promedio', 'Conteo'],
    description='Métrica:'
)
btn_temp = widgets.Button(description="Analizar", button_style='success')

# FILTRO
filter_checkbox = widgets.Checkbox(description="Activar filtro")
filter_column = widgets.Dropdown(description="Filtro:")
range_slider = widgets.FloatRangeSlider()

# ============================================================
# CARGAR VARIABLES
# ============================================================

def cargar_vars(b):
    
    with out_status:
        out_status.clear_output()
    
    if not df_valido():
        print("❌ Carga dataset primero")
        return
    
    cols = df.columns.tolist()
    
    var_uni.options = cols
    var_bi_x.options = cols
    var_bi_y.options = cols
    
    var_temp_target.options = df.select_dtypes(include=np.number).columns.tolist()
    filter_column.options = df.select_dtypes(include=np.number).columns.tolist()
    
    print("✅ Variables cargadas")

btn_load.on_click(cargar_vars)

# ============================================================
# FILTRO DINÁMICO (POKA-YOKE)
# ============================================================

def update_slider(change):
    
    if not df_valido():
        return
    
    col = filter_column.value
    if col is None:
        return
    
    min_val = df[col].min()
    max_val = df[col].max()
    
    # Ajuste automático según tipo
    if pd.api.types.is_integer_dtype(df[col]):
        range_slider.step = 1
    else:
        range_slider.step = (max_val - min_val) / 50
    
    range_slider.min = float(min_val)
    range_slider.max = float(max_val)
    range_slider.value = (float(min_val), float(max_val))

filter_column.observe(update_slider, names='value')

def aplicar_filtro(data):
    if filter_checkbox.value and filter_column.value:
        rmin, rmax = range_slider.value
        return data[(data[filter_column.value] >= rmin) & (data[filter_column.value] <= rmax)]
    return data

# ============================================================
# POKA-YOKE UNIVARIADO
# ============================================================

def update_uni(change):
    if not df_valido():
        return
    
    if plot_uni.value == 'Histograma':
        var_uni.options = df.select_dtypes(include=np.number).columns.tolist()
    else:
        var_uni.options = df.select_dtypes(exclude=np.number).columns.tolist()

plot_uni.observe(update_uni, names='value')

# ============================================================
# POKA-YOKE BIVARIADO
# ============================================================

def update_bi(change):
    if not df_valido():
        return
    
    if plot_bi.value == 'Scatter':
        cols = df.select_dtypes(include=np.number).columns.tolist()
        var_bi_x.options = cols
        var_bi_y.options = cols
    
    elif plot_bi.value == 'Boxplot':
        var_bi_x.options = df.select_dtypes(include=np.number).columns.tolist()
        var_bi_y.options = df.select_dtypes(exclude=np.number).columns.tolist()
    
    else:
        var_bi_x.options = df.select_dtypes(exclude=np.number).columns.tolist()
        var_bi_y.options = df.select_dtypes(include=np.number).columns.tolist()

plot_bi.observe(update_bi, names='value')

# ============================================================
# UNIVARIADO
# ============================================================

def run_uni(b):
    if not df_valido():
        return
    
    with out_uni:
        out_uni.clear_output()
        
        data = aplicar_filtro(df.copy())
        col = var_uni.value
        
        plt.figure(figsize=(6,4))
        
        if plot_uni.value == 'Histograma':
            sns.histplot(data[col], kde=True)
        else:
            data[col].value_counts().head(10).plot(kind='bar')
        
        plt.title(f"Univariado - {col}")
        plt.show()

btn_uni.on_click(run_uni)

# ============================================================
# BIVARIADO
# ============================================================

def run_bi(b):
    if not df_valido():
        return
    
    with out_bi:
        out_bi.clear_output()
        
        data = aplicar_filtro(df.copy())
        x = var_bi_x.value
        y = var_bi_y.value
        
        plt.figure(figsize=(6,4))
        
        if plot_bi.value == 'Scatter':
            sns.scatterplot(x=data[x], y=data[y])
        
        elif plot_bi.value == 'Boxplot':
            sns.boxplot(x=data[y], y=data[x])
        
        else:
            data.groupby(x)[y].mean().plot(kind='bar')
        
        plt.title(f"Bivariado - {x} vs {y}")
        plt.show()

btn_bi.on_click(run_bi)

# ============================================================
# MULTIVARIADO
# ============================================================

def run_multi(b):
    if not df_valido():
        return
    
    with out_multi:
        out_multi.clear_output()
        
        corr = df.select_dtypes(include=np.number).corr()
        
        plt.figure(figsize=(8,6))
        sns.heatmap(corr, cmap='coolwarm')
        plt.title("Correlación")
        plt.show()

btn_multi.on_click(run_multi)

# ============================================================
#  TEMPORAL PRO
# ============================================================

def run_temp(b):
    
    if not df_valido():
        return
    
    with out_temp:
        out_temp.clear_output()
        
        data = df.copy()
        target = var_temp_target.value
        
        if not pd.api.types.is_numeric_dtype(data[target]):
            print("⚠️ El target debe ser numérico")
            return
        
        if time_granularity.value == 'Año':
            fecha = 'arrival_date_year'
        elif time_granularity.value == 'Mes':
            fecha = 'arrival_date_month'
        elif time_granularity.value == 'Semana':
            fecha = 'arrival_date_week_number'
        else:
            fecha = 'arrival_date_day_of_month'
        
        if fecha == 'arrival_date_month':
            meses = ['January','February','March','April','May','June',
                     'July','August','September','October','November','December']
            data[fecha] = pd.Categorical(data[fecha], categories=meses, ordered=True)
        
        if agg_type.value == 'Promedio':
            temp = data.groupby(fecha)[target].mean().sort_index()
        else:
            temp = data.groupby(fecha)[target].count().sort_index()
        
        plt.figure(figsize=(7,4))
        temp.plot(marker='o')
        
        plt.title(f"{target} - Evolución ({time_granularity.value})")
        plt.xlabel(fecha)
        plt.ylabel(target)
        plt.grid()
        
        plt.show()

btn_temp.on_click(run_temp)

# ============================================================
# TABS
# ============================================================

tab_uni = widgets.VBox([widgets.HTML("<b> Univariado</b>"), var_uni, plot_uni, btn_uni, out_uni])
tab_bi = widgets.VBox([widgets.HTML("<b> Bivariado</b>"), var_bi_x, var_bi_y, plot_bi, btn_bi, out_bi])
tab_multi = widgets.VBox([widgets.HTML("<b> Correlación</b>"), btn_multi, out_multi])
tab_temp = widgets.VBox([
    widgets.HTML("<b> Temporal</b>"),
    time_granularity,
    agg_type,
    var_temp_target,
    btn_temp,
    out_temp
])

tabs = widgets.Tab(children=[tab_uni, tab_bi, tab_multi, tab_temp])
tabs.set_title(0, 'Univariado')
tabs.set_title(1, 'Bivariado')
tabs.set_title(2, 'Correlación')
tabs.set_title(3, 'Temporal')

# ============================================================
# UI FINAL
# ============================================================

display(widgets.VBox([
    widgets.HTML("<h2>EDA Interactivo Profesional</h2>"),
    btn_load,
    widgets.HTML("<b>Filtro global</b>"),
    filter_checkbox,
    filter_column,
    range_slider,
    tabs,
    out_status
]))


## ⚙️ Paso 2: Configuración de limpieza

Selecciona qué transformaciones deseas aplicar al dataset.

In [25]:
# ============================================================
# DATA CLEANING
# ============================================================

# ============================================================
# [1] IMPORTS
# ============================================================

import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from datetime import datetime

from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from scipy.stats import zscore

# ============================================================
# [2] VARIABLES GLOBALES
# ============================================================

df_clean = None
df_ready = None
df_feature_input = None
cleaning_confirmed = False

history_df = pd.DataFrame(columns=[
    "Paso", "Variable", "Método", "Recomendación", "Timestamp"
])

snapshots = []

treated_outliers = set()
treated_encoding = set()
treated_scaling = set()
duplicates_removed = False

out_hist = widgets.Output()
out_confirm_fe = widgets.Output()
out_rules = widgets.Output()

MONTH_MAP = {
    'January': 1, 'February': 2, 'March': 3, 'April': 4,
    'May': 5, 'June': 6, 'July': 7, 'August': 8,
    'September': 9, 'October': 10, 'November': 11, 'December': 12
}

# ============================================================
# [3] UTILIDADES
# ============================================================

def df_valido():
    return 'df' in globals() and isinstance(df, pd.DataFrame)

def save_snapshot():
    snapshots.append(df_clean.copy())

def render_history():
    with out_hist:
        out_hist.clear_output()
        display(history_df)

def log(paso, var, metodo, reco):
    global history_df

    decision = "✔" if metodo == reco else "⚠"

    history_df = pd.concat([
        history_df,
        pd.DataFrame({
            "Paso": [paso],
            "Variable": [var],
            "Método": [metodo],
            "Recomendación": [f"{reco} {decision}"],
            "Timestamp": [datetime.now().strftime("%H:%M:%S")]
        })
    ], ignore_index=True)

    render_history()

# ============================================================
# [4] RECOMENDACIONES
# ============================================================

def get_reco_null(col):
    if col == "children":
        return "median"
    if col == "country":
        return "mode"
    if col == "agent":
        return "median"
    if col == "company":
        return "eliminar_columna"

    data = df_clean[col]
    if pd.api.types.is_numeric_dtype(data):
        return "mean" if abs(data.skew()) < 0.5 else "median"
    return "mode"

def get_reco_out(col):
    if col == "adr":
        return ("Winsorizing", "útil para controlar extremos sin perder demasiadas filas")
    skew = df_clean[col].dropna().skew()
    return ("Z-score", "datos equilibrados") if abs(skew) < 0.5 else ("IQR", "valores extremos")

def get_reco_enc(col):
    if col == "arrival_date_month":
        return ("Ordinal", "tiene orden natural por meses")
    return ("OneHot", "pocas categorías") if df_clean[col].nunique() <= 10 else ("Ordinal", "muchas categorías")

def get_reco_scale(col):
    return ("Standard", "distribución normal") if abs(df_clean[col].skew()) < 0.5 else ("Robust", "outliers")

def get_exp_out(m):
    return {
        "IQR": "Elimina valores extremos usando rango intercuartílico.",
        "Z-score": "Elimina valores muy alejados del promedio.",
        "Winsorizing": "Recorta extremos sin eliminar filas."
    }[m]

def get_exp_enc(m):
    return {
        "OneHot": "Crea columnas binarias por categoría.",
        "Ordinal": "Convierte categorías a números."
    }[m]

def get_exp_scale(m):
    return {
        "Standard": "Centra y escala con media y desviación estándar.",
        "MinMax": "Lleva los valores a una escala entre 0 y 1.",
        "Robust": "Escala usando mediana y cuartiles, útil con outliers."
    }[m]

# ============================================================
# [5] RECONSTRUCCIÓN DE ESTADO
# ============================================================

def rebuild_treated_sets():
    global treated_outliers, treated_encoding, treated_scaling, duplicates_removed

    treated_outliers = set()
    treated_encoding = set()
    treated_scaling = set()
    duplicates_removed = False

    if df_clean is None or history_df.empty:
        return

    out_logged = history_df.loc[history_df["Paso"] == "Outliers", "Variable"].tolist()
    for col in out_logged:
        if col in df_clean.columns and pd.api.types.is_numeric_dtype(df_clean[col]):
            treated_outliers.add(col)

    enc_logged = history_df.loc[history_df["Paso"] == "Encoding", "Variable"].tolist()
    for col in enc_logged:
        if col not in df_clean.columns:
            treated_encoding.add(col)
        elif not pd.api.types.is_object_dtype(df_clean[col]):
            treated_encoding.add(col)

    scale_logged = history_df.loc[history_df["Paso"] == "Escalado", "Variable"].tolist()
    for col in scale_logged:
        if col in df_clean.columns and pd.api.types.is_numeric_dtype(df_clean[col]):
            treated_scaling.add(col)

    dup_logged = history_df.loc[history_df["Paso"] == "Duplicados", "Variable"].tolist()
    if len(dup_logged) > 0:
        duplicates_removed = True

# ============================================================
# [6] HELPERS UI
# ============================================================

def set_dropdown_value_safe(dropdown, preferred_value):
    options = list(dropdown.options) if dropdown.options is not None else []
    if not options:
        return
    if preferred_value in options:
        dropdown.value = preferred_value
    else:
        dropdown.value = options[0]

# ============================================================
# [7] CARGA Y REFRESH
# ============================================================

btn_load = widgets.Button(description="⚡ Cargar variables")

col_nulls = widgets.Dropdown(description="Col:")
col_out   = widgets.Dropdown(description="Col:")
col_enc   = widgets.Dropdown(description="Col:")
col_scale = widgets.Dropdown(description="Col:")

def refresh():
    if df_clean is None:
        return

    rebuild_treated_sets()

    prev_null = getattr(col_nulls, "value", None)
    prev_out = getattr(col_out, "value", None)
    prev_enc = getattr(col_enc, "value", None)
    prev_scale = getattr(col_scale, "value", None)

    null_candidates = [c for c in df_clean.columns if df_clean[c].isnull().sum() > 0]
    col_nulls.options = null_candidates if null_candidates else ["Sin nulos"]

    num_cols = df_clean.select_dtypes(include=np.number).columns.tolist()
    out_candidates = [c for c in num_cols if c not in treated_outliers]
    scale_candidates = [c for c in num_cols if c not in treated_scaling]

    col_out.options = out_candidates
    col_scale.options = scale_candidates

    obj_cols = df_clean.select_dtypes(include='object').columns.tolist()
    enc_candidates = [c for c in obj_cols if c not in treated_encoding]
    col_enc.options = enc_candidates

    set_dropdown_value_safe(col_nulls, prev_null if prev_null in list(col_nulls.options) else None)
    set_dropdown_value_safe(col_out, prev_out if prev_out in list(col_out.options) else None)
    set_dropdown_value_safe(col_enc, prev_enc if prev_enc in list(col_enc.options) else None)
    set_dropdown_value_safe(col_scale, prev_scale if prev_scale in list(col_scale.options) else None)

def cargar_vars(b):
    global df_clean, df_ready, df_feature_input, cleaning_confirmed, snapshots, history_df, duplicates_removed

    if not df_valido():
        print("❌ Cargar dataset primero")
        return

    df_clean = df.copy()
    df_ready = None
    df_feature_input = None
    cleaning_confirmed = False
    duplicates_removed = False
    snapshots = [df_clean.copy()]
    history_df = history_df.iloc[0:0].copy()

    refresh()
    render_history()

    with out_confirm_fe:
        out_confirm_fe.clear_output()

    with out_rules:
        out_rules.clear_output()

    update_dup_info()

    print("✅ Dataset listo")

btn_load.on_click(cargar_vars)

# ============================================================
# [8] REGLAS FIJAS
# ============================================================

guide_rules = widgets.HTML("""
<b>🎯 RECOMENDACIÓN</b><br>
Aplicar primero estas reglas de negocio:<br><br>
1. Eliminar columnas con leakage: <b>reservation_status</b>, <b>reservation_status_date</b><br>
2. Eliminar <b>company</b> por alta proporción de nulos<br>
3. Corregir <b>children == 10 → 0</b><br>
4. Eliminar reservas sin personas<br>
5. Eliminar reservas sin noches<br>
6. Crear <b>arrival_date_month_num</b><br><br>
<b>📘 EXPLICACIÓN</b><br>
Estas reglas obedecen al analisis de negocio.
""")

btn_rules = widgets.Button(description="Aplicar reglas de negocio", button_style='warning')

def aplicar_reglas_equipo(b):
    global df_clean, cleaning_confirmed, df_feature_input

    if df_clean is None:
        return

    save_snapshot()

    with out_rules:
        out_rules.clear_output()

        cols_drop = [c for c in ['reservation_status', 'reservation_status_date', 'company'] if c in df_clean.columns]
        if cols_drop:
            df_clean = df_clean.drop(columns=cols_drop)

        if 'children' in df_clean.columns:
            df_clean['children'] = df_clean['children'].fillna(0)
            df_clean.loc[df_clean['children'] == 10, 'children'] = 0

        if all(c in df_clean.columns for c in ['adults', 'children', 'babies']):
            mask_sin_personas = (
                (df_clean['adults'] == 0) &
                (df_clean['children'].fillna(0) == 0) &
                (df_clean['babies'] == 0)
            )
            n_sin_personas = int(mask_sin_personas.sum())
            df_clean = df_clean[~mask_sin_personas].copy()
        else:
            n_sin_personas = 0

        if all(c in df_clean.columns for c in ['stays_in_week_nights', 'stays_in_weekend_nights']):
            mask_sin_noches = (
                (df_clean['stays_in_week_nights'] == 0) &
                (df_clean['stays_in_weekend_nights'] == 0)
            )
            n_sin_noches = int(mask_sin_noches.sum())
            df_clean = df_clean[~mask_sin_noches].copy()
        else:
            n_sin_noches = 0

        if 'arrival_date_month' in df_clean.columns and 'arrival_date_month_num' not in df_clean.columns:
            df_clean['arrival_date_month_num'] = df_clean['arrival_date_month'].map(MONTH_MAP)

        if 'children' in df_clean.columns:
            if pd.api.types.is_float_dtype(df_clean['children']) or pd.api.types.is_integer_dtype(df_clean['children']):
                df_clean['children'] = df_clean['children'].astype(int)

        cleaning_confirmed = False
        df_feature_input = None

        log("Reglas fijas", "Dataset", "aplicar_reglas_equipo", "aplicar_reglas_equipo")
        refresh()
        update_dup_info()

        print(f"Columnas eliminadas: {cols_drop}")
        print(f"Reservas sin personas eliminadas: {n_sin_personas}")
        print(f"Reservas sin noches eliminadas: {n_sin_noches}")
        print(f"Shape actual: {df_clean.shape}")

btn_rules.on_click(aplicar_reglas_equipo)

# ============================================================
# [9] NULOS
# ============================================================

guide_nulls = widgets.HTML()
mechanism_nulls = widgets.RadioButtons(options=['MCAR', 'MAR', 'MNAR'])
method_nulls = widgets.Dropdown()
btn_nulls = widgets.Button(description="Aplicar")

def update_nulls(change=None):
    col = col_nulls.value
    if col in [None, "Sin nulos"] or df_clean is None:
        guide_nulls.value = ""
        method_nulls.options = []
        return

    reco = get_reco_null(col)

    if col == "children":
        method_nulls.options = ['median', 'mean', 'mode']
        exp = "En este proyecto también puede corregirse manualmente si hay valores imposibles como 10."
    elif col == "country":
        method_nulls.options = ['mode']
        exp = "Country suele imputarse con la moda por bajo porcentaje de nulos."
    elif col == "agent":
        method_nulls.options = ['median', 'mean']
        exp = "Agent puede imputarse numéricamente."
    else:
        if mechanism_nulls.value == 'MCAR':
            if pd.api.types.is_numeric_dtype(df_clean[col]):
                method_nulls.options = ['mean', 'median']
            else:
                method_nulls.options = ['mode']
            exp = "Faltantes aleatorios → imputar."
        elif mechanism_nulls.value == 'MAR':
            if pd.api.types.is_numeric_dtype(df_clean[col]):
                method_nulls.options = ['KNN', 'Iterative']
            else:
                method_nulls.options = ['mode']
            exp = "Dependen de otras variables."
        else:
            method_nulls.options = ['flag']
            exp = "No imputar → crear indicador."

    if reco in list(method_nulls.options):
        method_nulls.value = reco
    else:
        method_nulls.value = method_nulls.options[0]

    if pd.api.types.is_numeric_dtype(df_clean[col]):
        motivo = "distribución simétrica" if reco == "mean" else "distribución sesgada"
    else:
        motivo = "variable categórica"

    guide_nulls.value = f"""
<b>🎯 RECOMENDACIÓN</b><br>{reco} → {motivo}<br><br>
<b>📘 EXPLICACIÓN DE ESTA OPCIÓN</b><br>{exp}
"""

col_nulls.observe(update_nulls, names='value')
mechanism_nulls.observe(update_nulls, names='value')

def aplicar_nulls(b):
    global df_clean, cleaning_confirmed, df_feature_input

    col = col_nulls.value
    if col in [None, "Sin nulos"]:
        return

    reco_real = get_reco_null(col)
    save_snapshot()

    if method_nulls.value == 'mean':
        df_clean[col] = df_clean[col].fillna(df_clean[col].mean())
    elif method_nulls.value == 'median':
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())
    elif method_nulls.value == 'mode':
        df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])
    elif method_nulls.value == 'KNN':
        df_clean[[col]] = KNNImputer().fit_transform(df_clean[[col]])
    elif method_nulls.value == 'Iterative':
        df_clean[[col]] = IterativeImputer().fit_transform(df_clean[[col]])
    else:
        df_clean[col + "_flag"] = df_clean[col].isnull().astype(int)

    if col == "children":
        df_clean.loc[df_clean['children'] == 10, 'children'] = 0
        if pd.api.types.is_numeric_dtype(df_clean['children']):
            df_clean['children'] = df_clean['children'].astype(int)

    cleaning_confirmed = False
    df_feature_input = None

    log("Nulos", col, method_nulls.value, reco_real)
    refresh()
    update_nulls()
    update_dup_info()

btn_nulls.on_click(aplicar_nulls)

# ============================================================
# [10] OUTLIERS
# ============================================================

guide_out = widgets.HTML()
method_out = widgets.Dropdown(options=['IQR', 'Z-score', 'Winsorizing'])
btn_out = widgets.Button(description="Aplicar")

def update_out(change=None):
    col = col_out.value
    if col is None or df_clean is None:
        guide_out.value = ""
        return

    reco, motivo = get_reco_out(col)

    guide_out.value = f"""
<b>🎯 RECOMENDACIÓN</b><br>{reco} → {motivo}<br><br>
<b>📘 EXPLICACIÓN</b><br>{get_exp_out(method_out.value)}
"""

col_out.observe(update_out, names='value')
method_out.observe(update_out, names='value')

def aplicar_out(b):
    global df_clean, cleaning_confirmed, df_feature_input

    col = col_out.value
    if col is None:
        return

    reco_real, _ = get_reco_out(col)
    save_snapshot()

    if col == "adr" and method_out.value == "Winsorizing":
        # criterio más cercano al notebook del equipo
        df_clean = df_clean[df_clean['adr'] >= 0].copy()
        p99 = df_clean['adr'].quantile(0.99)
        df_clean['adr'] = df_clean['adr'].clip(upper=p99)
    else:
        if method_out.value == "IQR":
            Q1, Q3 = df_clean[col].quantile([0.25, 0.75])
            IQR = Q3 - Q1
            df_clean = df_clean[
                (df_clean[col] >= Q1 - 1.5 * IQR) &
                (df_clean[col] <= Q3 + 1.5 * IQR)
            ].copy()
        elif method_out.value == "Z-score":
            z = zscore(df_clean[col].dropna())
            mask = np.ones(len(df_clean), dtype=bool)
            mask[df_clean[col].dropna().index] = np.abs(z) < 3
            df_clean = df_clean[mask].copy()
        else:
            p99 = df_clean[col].quantile(0.99)
            df_clean[col] = df_clean[col].clip(upper=p99)

    cleaning_confirmed = False
    df_feature_input = None

    log("Outliers", col, method_out.value, reco_real)
    refresh()
    update_out()
    update_dup_info()

btn_out.on_click(aplicar_out)

# ============================================================
# [11] DUPLICADOS
# ============================================================

guide_dup = widgets.HTML()
btn_dup = widgets.Button(description="Eliminar duplicados exactos", button_style='warning')

def update_dup_info():
    if df_clean is None:
        guide_dup.value = ""
        return

    n_dup = int(df_clean.duplicated().sum())

    if n_dup > 0:
        guide_dup.value = f"""
<b>🎯 RECOMENDACIÓN</b><br>drop_duplicates → hay {n_dup} filas duplicadas exactas<br><br>
<b>📘 EXPLICACIÓN</b><br>Elimina filas completamente idénticas en todas las columnas.
"""
    else:
        guide_dup.value = f"""
<b>🎯 RECOMENDACIÓN</b><br>No hay duplicados exactos pendientes<br><br>
<b>📘 EXPLICACIÓN</b><br>El dataset ya no contiene filas completamente idénticas.
"""

def aplicar_dup(b):
    global df_clean, cleaning_confirmed, df_feature_input

    if df_clean is None:
        return

    n_antes = len(df_clean)
    n_dup = int(df_clean.duplicated().sum())

    if n_dup == 0:
        update_dup_info()
        return

    save_snapshot()

    df_clean = df_clean.drop_duplicates().reset_index(drop=True)

    cleaning_confirmed = False
    df_feature_input = None

    n_despues = len(df_clean)
    filas_eliminadas = n_antes - n_despues

    log("Duplicados", "Fila completa", "drop_duplicates", "drop_duplicates")
    refresh()
    update_dup_info()

    print(f"Filas eliminadas: {filas_eliminadas}")
    print(f"Shape: {df_clean.shape}")

btn_dup.on_click(aplicar_dup)

# ============================================================
# [12] ENCODING
# ============================================================

guide_enc = widgets.HTML()
method_enc = widgets.Dropdown(options=['OneHot', 'Ordinal'])
btn_enc = widgets.Button(description="Aplicar")

def update_enc(change=None):
    col = col_enc.value
    if col is None or df_clean is None:
        guide_enc.value = ""
        return

    reco, motivo = get_reco_enc(col)

    guide_enc.value = f"""
<b>🎯 RECOMENDACIÓN</b><br>{reco} → {motivo}<br><br>
<b>📘 EXPLICACIÓN</b><br>{get_exp_enc(method_enc.value)}
"""

col_enc.observe(update_enc, names='value')
method_enc.observe(update_enc, names='value')

def aplicar_enc(b):
    global df_clean, cleaning_confirmed, df_feature_input

    col = col_enc.value
    if col is None:
        return

    if df_clean[col].isnull().sum() > 0:
        print("⚠️ Trata nulos primero")
        return

    reco_real, _ = get_reco_enc(col)
    save_snapshot()

    if method_enc.value == 'OneHot':
        df_clean = pd.get_dummies(df_clean, columns=[col], drop_first=False)
    else:
        if col == "arrival_date_month":
            df_clean[col] = df_clean[col].map(MONTH_MAP)
        else:
            df_clean[col] = pd.Categorical(df_clean[col]).codes

    cleaning_confirmed = False
    df_feature_input = None

    log("Encoding", col, method_enc.value, reco_real)
    refresh()
    update_enc()
    update_dup_info()

btn_enc.on_click(aplicar_enc)

# ============================================================
# [13] ESCALADO
# ============================================================

guide_scale = widgets.HTML()
method_scale = widgets.Dropdown(options=['Standard', 'MinMax', 'Robust'])
btn_scale = widgets.Button(description="Aplicar")

def update_scale(change=None):
    col = col_scale.value
    if col is None or df_clean is None:
        guide_scale.value = ""
        return

    reco, motivo = get_reco_scale(col)

    guide_scale.value = f"""
<b>🎯 RECOMENDACIÓN</b><br>{reco} → {motivo}<br><br>
<b>📘 EXPLICACIÓN</b><br>{get_exp_scale(method_scale.value)}
"""

col_scale.observe(update_scale, names='value')
method_scale.observe(update_scale, names='value')

def aplicar_scale(b):
    global df_clean, cleaning_confirmed, df_feature_input

    col = col_scale.value
    if col is None:
        return

    reco_real, _ = get_reco_scale(col)
    save_snapshot()

    scaler = {
        'Standard': StandardScaler(),
        'MinMax': MinMaxScaler(),
        'Robust': RobustScaler()
    }[method_scale.value]

    df_clean[[col]] = scaler.fit_transform(df_clean[[col]])

    cleaning_confirmed = False
    df_feature_input = None

    log("Escalado", col, method_scale.value, reco_real)
    refresh()
    update_scale()
    update_dup_info()

btn_scale.on_click(aplicar_scale)

# ============================================================
# [14] DESHACER
# ============================================================

btn_undo = widgets.Button(description="Deshacer")

def undo(b):
    global df_clean, history_df, cleaning_confirmed, df_feature_input

    if len(snapshots) <= 1:
        return

    snapshots.pop()
    df_clean = snapshots[-1].copy()
    history_df = history_df.iloc[:-1].copy()

    cleaning_confirmed = False
    df_feature_input = None

    refresh()
    render_history()

    update_nulls()
    update_out()
    update_dup_info()
    update_enc()
    update_scale()

btn_undo.on_click(undo)

# ============================================================
# [15] FINALIZAR CLEANING
# ============================================================

btn_finish = widgets.Button(description="Finalizar cleaning", button_style='success')

def finalizar(b):
    global df_ready

    if df_clean.isnull().sum().sum() > 0:
        print("❌ Hay nulos")
        return

    if len(df_clean.select_dtypes(include='object')) > 0:
        print("❌ Falta encoding")
        return

    df_ready = df_clean.copy()
    print("🎯 Dataset listo")

btn_finish.on_click(finalizar)

# ============================================================
# [16] CONFIRMAR PASO A FEATURE ENGINEERING
# ============================================================

btn_confirm_fe = widgets.Button(
    description="Confirmar para Feature Engineering",
    button_style='info'
)

def confirmar_feature_engineering(b):
    global df_feature_input, cleaning_confirmed

    with out_confirm_fe:
        out_confirm_fe.clear_output()

        if df_clean is None:
            print("❌ No hay dataset cargado")
            return

        if df_clean.shape[0] == 0 or df_clean.shape[1] == 0:
            print("❌ El dataset está vacío")
            return

        if df_clean.isnull().sum().sum() > 0:
            print("❌ Aún hay valores nulos")
            return

        obj_cols = df_clean.select_dtypes(include='object').columns.tolist()
        if len(obj_cols) > 0:
            print("❌ Aún hay columnas categóricas sin procesar:")
            print(obj_cols)
            return

        duplicated_rows = int(df_clean.duplicated().sum())
        if duplicated_rows > 0:
            print(f"❌ Aún hay {duplicated_rows} filas duplicadas")
            return

        df_feature_input = df_clean.copy()
        cleaning_confirmed = True

        print("✅ Dataset confirmado para Feature Engineering")
        print(f"Filas: {df_feature_input.shape[0]}")
        print(f"Columnas: {df_feature_input.shape[1]}")

btn_confirm_fe.on_click(confirmar_feature_engineering)

# ============================================================
# [17] UI FINAL
# ============================================================

tabs = widgets.Tab(children=[
    widgets.VBox([guide_rules, btn_rules, out_rules]),
    widgets.VBox([col_nulls, guide_nulls, mechanism_nulls, method_nulls, btn_nulls]),
    widgets.VBox([col_out, guide_out, method_out, btn_out]),
    widgets.VBox([guide_dup, btn_dup]),
    widgets.VBox([col_enc, guide_enc, method_enc, btn_enc]),
    widgets.VBox([col_scale, guide_scale, method_scale, btn_scale])
])

for i, t in enumerate(["Reglas fijas", "Nulos", "Outliers", "Duplicados", "Encoding", "Escalado"]):
    tabs.set_title(i, t)

display(widgets.VBox([
    btn_load,
    tabs,
    widgets.HTML("<h3>Historial</h3>"),
    out_hist,
    widgets.HBox([btn_undo, btn_finish]),
    widgets.HTML("<h3>Paso a Feature Engineering</h3>"),
    btn_confirm_fe,
    out_confirm_fe
]))

## Paso 3: Feature Engineering

Selecciona las transformaciones para generar nuevas variables.

In [26]:
# ============================================================
# FEATURE ENGINEERING PLATFORM
# ============================================================

# ============================================================
# [1] IMPORTS
# ============================================================

import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from datetime import datetime

import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# [2] VARIABLES GLOBALES
# ============================================================

df_fe = None
df_balance_input = None
feature_confirmed = False

history_fe = pd.DataFrame(columns=[
    "Paso", "Variable", "Método", "Recomendación", "Timestamp"
])

snapshots_fe = []

out_hist_fe = widgets.Output()
out_confirm_balance = widgets.Output()
out_selection_analysis = widgets.Output()

selection_analysis_done = False

MONTH_MAP = {
    'January': 1, 'February': 2, 'March': 3, 'April': 4,
    'May': 5, 'June': 6, 'July': 7, 'August': 8,
    'September': 9, 'October': 10, 'November': 11, 'December': 12
}

# ============================================================
# [3] UTILIDADES
# ============================================================

def fe_valido():
    return 'df_feature_input' in globals() and isinstance(df_feature_input, pd.DataFrame)

def save_snapshot_fe():
    snapshots_fe.append(df_fe.copy())

def render_history_fe():
    with out_hist_fe:
        out_hist_fe.clear_output()
        display(history_fe)

def log_fe(paso, var, metodo, reco):
    global history_fe
    decision = "✔" if metodo == reco else "⚠"

    history_fe = pd.concat([
        history_fe,
        pd.DataFrame({
            "Paso": [paso],
            "Variable": [var],
            "Método": [metodo],
            "Recomendación": [f"{reco} {decision}"],
            "Timestamp": [datetime.now().strftime("%H:%M:%S")]
        })
    ], ignore_index=True)

    render_history_fe()

def invalid_widget_value(x):
    return x is None or (isinstance(x, float) and pd.isna(x))

# ============================================================
# [4] HELPERS
# ============================================================

def get_numeric_cols():
    if df_fe is None:
        return []
    return df_fe.select_dtypes(include=np.number).columns.tolist()

def get_all_cols():
    if df_fe is None:
        return []
    return df_fe.columns.tolist()

# ============================================================
# [5] TABLAS DE SUGERENCIAS
# ============================================================

def get_direct_recommendation_table():
    return """
<table style="border-collapse: collapse; width: 100%; font-size: 13px;">
  <tr>
    <th style="border:1px solid #ccc; padding:6px; text-align:left;">Feature sugerida</th>
    <th style="border:1px solid #ccc; padding:6px; text-align:left;">Método</th>
    <th style="border:1px solid #ccc; padding:6px; text-align:left;">Columnas sugeridas</th>
    <th style="border:1px solid #ccc; padding:6px; text-align:left;">Justificación</th>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:6px;">total_nights</td>
    <td style="border:1px solid #ccc; padding:6px;">suma</td>
    <td style="border:1px solid #ccc; padding:6px;">stays_in_week_nights + stays_in_weekend_nights</td>
    <td style="border:1px solid #ccc; padding:6px;">La duración total de la estadía puede influir en la cancelación.</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:6px;">total_guests</td>
    <td style="border:1px solid #ccc; padding:6px;">suma</td>
    <td style="border:1px solid #ccc; padding:6px;">adults + children + babies</td>
    <td style="border:1px solid #ccc; padding:6px;">Resume el tamaño total del grupo.</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:6px;">total_price</td>
    <td style="border:1px solid #ccc; padding:6px;">producto</td>
    <td style="border:1px solid #ccc; padding:6px;">adr × total_nights</td>
    <td style="border:1px solid #ccc; padding:6px;">Aproxima el valor económico total de la reserva.</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:6px;">is_loyal</td>
    <td style="border:1px solid #ccc; padding:6px;">binaria</td>
    <td style="border:1px solid #ccc; padding:6px;">is_repeated_guest == 1</td>
    <td style="border:1px solid #ccc; padding:6px;">Captura fidelidad del cliente.</td>
  </tr>
</table>
"""

def get_ratio_recommendation_table():
    return """
<table style="border-collapse: collapse; width: 100%; font-size: 13px;">
  <tr>
    <th style="border:1px solid #ccc; padding:6px; text-align:left;">Feature sugerida</th>
    <th style="border:1px solid #ccc; padding:6px; text-align:left;">Operación</th>
    <th style="border:1px solid #ccc; padding:6px; text-align:left;">Columnas sugeridas</th>
    <th style="border:1px solid #ccc; padding:6px; text-align:left;">Justificación</th>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:6px;">price_per_person</td>
    <td style="border:1px solid #ccc; padding:6px;">división</td>
    <td style="border:1px solid #ccc; padding:6px;">adr / total_guests</td>
    <td style="border:1px solid #ccc; padding:6px;">Permite analizar el costo individual de la reserva.</td>
  </tr>
</table>
"""

def get_temporal_recommendation_table():
    return """
<table style="border-collapse: collapse; width: 100%; font-size: 13px;">
  <tr>
    <th style="border:1px solid #ccc; padding:6px; text-align:left;">Variable sugerida</th>
    <th style="border:1px solid #ccc; padding:6px; text-align:left;">Transformar</th>
    <th style="border:1px solid #ccc; padding:6px; text-align:left;">Categorizar</th>
    <th style="border:1px solid #ccc; padding:6px; text-align:left;">Justificación</th>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:6px;">arrival_date_month</td>
    <td style="border:1px solid #ccc; padding:6px;">arrival_month_num</td>
    <td style="border:1px solid #ccc; padding:6px;">is_high_season (July, August)</td>
    <td style="border:1px solid #ccc; padding:6px;">Permite representar el mes en formato numérico y luego resumir alta temporada.</td>
  </tr>
</table>
"""

def get_diff_recommendation_table():
    return """
<table style="border-collapse: collapse; width: 100%; font-size: 13px;">
  <tr>
    <th style="border:1px solid #ccc; padding:6px; text-align:left;">Observación</th>
    <th style="border:1px solid #ccc; padding:6px; text-align:left;">Uso</th>
    <th style="border:1px solid #ccc; padding:6px; text-align:left;">Justificación</th>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:6px;">No se priorizó una diferencia específica en el script base.</td>
    <td style="border:1px solid #ccc; padding:6px;">Opcional</td>
    <td style="border:1px solid #ccc; padding:6px;">Puedes usar esta sección si identificas una brecha relevante entre dos variables numéricas.</td>
  </tr>
</table>
"""

def get_bin_recommendation_table():
    return """
<table style="border-collapse: collapse; width: 100%; font-size: 13px;">
  <tr>
    <th style="border:1px solid #ccc; padding:6px; text-align:left;">Feature sugerida</th>
    <th style="border:1px solid #ccc; padding:6px; text-align:left;">Variable sugerida</th>
    <th style="border:1px solid #ccc; padding:6px; text-align:left;">Valores sugeridos</th>
    <th style="border:1px solid #ccc; padding:6px; text-align:left;">Rangos sugeridos</th>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:6px;">lead_time_group</td>
    <td style="border:1px solid #ccc; padding:6px;">lead_time</td>
    <td style="border:1px solid #ccc; padding:6px;">last_minute, short, medium, long, very_long</td>
    <td style="border:1px solid #ccc; padding:6px;">0-7, 7-30, 30-90, 90-180, 180-700</td>
  </tr>
</table>
"""

def get_inter_recommendation_table():
    return """
<table style="border-collapse: collapse; width: 100%; font-size: 13px;">
  <tr>
    <th style="border:1px solid #ccc; padding:6px; text-align:left;">Feature sugerida</th>
    <th style="border:1px solid #ccc; padding:6px; text-align:left;">Método</th>
    <th style="border:1px solid #ccc; padding:6px; text-align:left;">Columnas sugeridas</th>
    <th style="border:1px solid #ccc; padding:6px; text-align:left;">Justificación</th>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:6px;">price_per_person_total</td>
    <td style="border:1px solid #ccc; padding:6px;">división</td>
    <td style="border:1px solid #ccc; padding:6px;">total_price / total_guests</td>
    <td style="border:1px solid #ccc; padding:6px;">Analiza el costo individual considerando la duración de la estadía.</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:6px;">lead_time_x_nights</td>
    <td style="border:1px solid #ccc; padding:6px;">multiplicación</td>
    <td style="border:1px solid #ccc; padding:6px;">lead_time × total_nights</td>
    <td style="border:1px solid #ccc; padding:6px;">Combina anticipación con duración de la estadía.</td>
  </tr>
</table>
"""

def get_trans_recommendation_table():
    return """
<table style="border-collapse: collapse; width: 100%; font-size: 13px;">
  <tr>
    <th style="border:1px solid #ccc; padding:6px; text-align:left;">Feature sugerida</th>
    <th style="border:1px solid #ccc; padding:6px; text-align:left;">Método</th>
    <th style="border:1px solid #ccc; padding:6px; text-align:left;">Columna sugerida</th>
    <th style="border:1px solid #ccc; padding:6px; text-align:left;">Justificación</th>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:6px;">adr_log</td>
    <td style="border:1px solid #ccc; padding:6px;">log</td>
    <td style="border:1px solid #ccc; padding:6px;">adr</td>
    <td style="border:1px solid #ccc; padding:6px;">Reduce la asimetría y el impacto de valores extremos.</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:6px;">lead_time_log</td>
    <td style="border:1px solid #ccc; padding:6px;">log</td>
    <td style="border:1px solid #ccc; padding:6px;">lead_time</td>
    <td style="border:1px solid #ccc; padding:6px;">Comprime la cola larga observada en la distribución.</td>
  </tr>
</table>
"""

def get_selection_recommendation_table():
    return """
<table style="border-collapse: collapse; width: 100%; font-size: 13px;">
  <tr>
    <th style="border:1px solid #ccc; padding:6px; text-align:left;">Tipo</th>
    <th style="border:1px solid #ccc; padding:6px; text-align:left;">Sugerencia</th>
    <th style="border:1px solid #ccc; padding:6px; text-align:left;">Justificación</th>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:6px;">Redundancia</td>
    <td style="border:1px solid #ccc; padding:6px;">Eliminar is_repeated_guest, arrival_date_week_number, stays_in_week_nights, total_price</td>
    <td style="border:1px solid #ccc; padding:6px;">Fueron identificadas como redundantes respecto a nuevas features o variables altamente correlacionadas.</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:6px;">Prefijos</td>
    <td style="border:1px solid #ccc; padding:6px;">Eliminar reserved_room_type_* y assigned_room_type_*</td>
    <td style="border:1px solid #ccc; padding:6px;">Se eliminaron por redundancia y simplificación del espacio de variables.</td>
  </tr>
</table>
"""

# ============================================================
# [6] REFRESH
# ============================================================

def refresh_fe():
    if df_fe is None:
        return

    num_cols = get_numeric_cols()
    all_cols = get_all_cols()

    direct_col1.options = all_cols
    direct_col2.options = ["(ninguna)"] + all_cols
    direct_col3.options = ["(ninguna)"] + all_cols

    ratio_col1.options = num_cols
    ratio_col2.options = num_cols

    diff_col1.options = num_cols
    diff_col2.options = num_cols

    temporal_col.options = all_cols

    bin_col.options = num_cols

    inter_col1.options = num_cols
    inter_col2.options = num_cols

    trans_col.options = num_cols

# ============================================================
# [7] CARGA DESDE CLEANING
# ============================================================

btn_load_fe = widgets.Button(description="Analizar data Cleaning", button_style='warning')

def cargar_feature_data(b):
    global df_fe, df_balance_input, feature_confirmed, history_fe, snapshots_fe, selection_analysis_done

    if not fe_valido():
        print("❌ No encuentro df_feature_input. Primero confirma el paso desde Cleaning.")
        return

    df_fe = df_feature_input.copy()
    df_balance_input = None
    feature_confirmed = False
    selection_analysis_done = False
    history_fe = history_fe.iloc[0:0].copy()
    snapshots_fe = [df_fe.copy()]

    refresh_fe()
    render_history_fe()

    with out_confirm_balance:
        out_confirm_balance.clear_output()

    with out_selection_analysis:
        out_selection_analysis.clear_output()

    update_direct_table()
    update_ratio_table()
    update_temporal_table()
    update_diff_table()
    update_bin_table()
    update_inter_table()
    update_trans_table()
    update_sel_table()
    update_temporal_visibility()
    update_inter_visibility()
    update_trans_visibility()
    update_sel_visibility()
    autofill_trans_name()

    print("✅ Dataset cargado desde Cleaning")

btn_load_fe.on_click(cargar_feature_data)

# ============================================================
# [8] TAB 1 - DERIVADAS
# ============================================================

direct_table = widgets.HTML()
direct_method = widgets.Dropdown(options=['suma', 'producto', 'binaria'], description="Método:")
direct_col1 = widgets.Dropdown(description="Col 1:")
direct_col2 = widgets.Dropdown(description="Col 2:")
direct_col3 = widgets.Dropdown(description="Col 3:")
direct_name = widgets.Text(description="Nombre:")
direct_binary_value = widgets.Text(description="Valor == ", value="1")
direct_binary_box = widgets.VBox()
btn_direct = widgets.Button(description="Aplicar")

def update_direct_table():
    direct_table.value = f"""
<b> SUGERENCIAS DE DERIVADAS</b><br><br>
{get_direct_recommendation_table()}
"""
    direct_binary_box.children = [direct_binary_value] if direct_method.value == "binaria" else []

direct_method.observe(lambda change: update_direct_table(), names='value')

def aplicar_direct(b):
    global df_fe, feature_confirmed, df_balance_input

    c1 = direct_col1.value
    c2 = direct_col2.value
    c3 = direct_col3.value
    name = direct_name.value.strip()

    if df_fe is None or invalid_widget_value(c1) or name == "":
        return

    save_snapshot_fe()

    if direct_method.value == "suma":
        if c2 == "(ninguna)" or invalid_widget_value(c2):
            print("⚠️ Para suma debes elegir una Col 2 válida.")
            snapshots_fe.pop()
            return
        df_fe[name] = df_fe[c1] + df_fe[c2]
        if c3 != "(ninguna)" and not invalid_widget_value(c3):
            df_fe[name] = df_fe[name] + df_fe[c3]

    elif direct_method.value == "producto":
        if c2 == "(ninguna)" or invalid_widget_value(c2):
            print("⚠️ Para producto debes elegir una Col 2 válida.")
            snapshots_fe.pop()
            return
        df_fe[name] = df_fe[c1] * df_fe[c2]
        if c3 != "(ninguna)" and not invalid_widget_value(c3):
            df_fe[name] = df_fe[name] * df_fe[c3]

    else:
        try:
            val = float(direct_binary_value.value)
            if val.is_integer():
                val = int(val)
        except:
            val = direct_binary_value.value
        df_fe[name] = (df_fe[c1] == val).astype(int)

    feature_confirmed = False
    df_balance_input = None

    log_fe("Derivada", f"{c1} | {c2} | {c3}", direct_method.value, "manual")
    refresh_fe()
    update_direct_table()

btn_direct.on_click(aplicar_direct)

# ============================================================
# [9] TAB 2 - RATIOS
# ============================================================

ratio_table = widgets.HTML()
ratio_col1 = widgets.Dropdown(description="Col 1:")
ratio_col2 = widgets.Dropdown(description="Col 2:")
ratio_name = widgets.Text(description="Nombre:")
btn_ratio = widgets.Button(description="Aplicar")

def update_ratio_table():
    ratio_table.value = f"""
<b> SUGERENCIAS DE RATIOS</b><br><br>
{get_ratio_recommendation_table()}
"""

def aplicar_ratio(b):
    global df_fe, feature_confirmed, df_balance_input

    c1 = ratio_col1.value
    c2 = ratio_col2.value
    name = ratio_name.value.strip()

    if df_fe is None or invalid_widget_value(c1) or invalid_widget_value(c2) or name == "":
        return

    if c1 == c2:
        print("⚠️ Elige columnas distintas")
        return

    save_snapshot_fe()
    df_fe[name] = df_fe[c1] / df_fe[c2].replace(0, np.nan)

    feature_confirmed = False
    df_balance_input = None

    log_fe("Ratio", f"{c1} | {c2}", name, "manual")
    refresh_fe()
    update_ratio_table()

btn_ratio.on_click(aplicar_ratio)

# ============================================================
# [10] TAB 3 - TEMPORALES
# ============================================================

temporal_table = widgets.HTML()
temporal_col = widgets.Dropdown(description="Variable:")
temporal_transform = widgets.Checkbox(value=True, description="Transformar a número")
temporal_name_num = widgets.Text(description="Nombre num:", value="arrival_month_num")
temporal_categorize = widgets.Checkbox(value=False, description="Categorizar")
high_months = widgets.Text(description="Clases:", value="July,August")
temporal_name_cat = widgets.Text(description="Categoría:", value="is_high_season")
temporal_dynamic_box = widgets.VBox()
btn_temporal = widgets.Button(description="Aplicar")

def update_temporal_table():
    temporal_table.value = f"""
<b>🎯 SUGERENCIAS TEMPORALES</b><br><br>
{get_temporal_recommendation_table()}
"""

def update_temporal_visibility(change=None):
    children = [temporal_col, temporal_transform]

    if temporal_transform.value:
        children.append(temporal_name_num)

    children.append(temporal_categorize)

    if temporal_categorize.value:
        children.extend([high_months, temporal_name_cat])

    temporal_dynamic_box.children = children

temporal_transform.observe(update_temporal_visibility, names='value')
temporal_categorize.observe(update_temporal_visibility, names='value')

def aplicar_temporal(b):
    global df_fe, feature_confirmed, df_balance_input

    col = temporal_col.value

    if df_fe is None or invalid_widget_value(col):
        return

    if not temporal_transform.value and not temporal_categorize.value:
        print("⚠️ Marca al menos una acción: transformar o categorizar.")
        return

    save_snapshot_fe()

    if temporal_transform.value:
        name_num = temporal_name_num.value.strip()
        if name_num == "":
            snapshots_fe.pop()
            return
        df_fe[name_num] = df_fe[col].map(MONTH_MAP)

    if temporal_categorize.value:
        name_cat = temporal_name_cat.value.strip()
        if name_cat == "":
            snapshots_fe.pop()
            return
        months = [x.strip() for x in high_months.value.split(",") if x.strip() != ""]
        df_fe[name_cat] = df_fe[col].isin(months).astype(int)

    feature_confirmed = False
    df_balance_input = None

    metodo = []
    if temporal_transform.value:
        metodo.append("transformar")
    if temporal_categorize.value:
        metodo.append("categorizar")

    log_fe("Temporal", col, " + ".join(metodo), "manual")
    refresh_fe()
    update_temporal_table()
    update_temporal_visibility()

btn_temporal.on_click(aplicar_temporal)

# ============================================================
# [11] TAB 4 - DIFERENCIAS
# ============================================================

diff_table = widgets.HTML()
diff_col1 = widgets.Dropdown(description="Col 1:")
diff_col2 = widgets.Dropdown(description="Col 2:")
diff_name = widgets.Text(description="Nombre:")
btn_diff = widgets.Button(description="Aplicar")

def update_diff_table():
    diff_table.value = f"""
<b> SUGERENCIAS DE DIFERENCIAS</b><br><br>
{get_diff_recommendation_table()}
"""

def aplicar_diff(b):
    global df_fe, feature_confirmed, df_balance_input

    c1 = diff_col1.value
    c2 = diff_col2.value
    name = diff_name.value.strip()

    if df_fe is None or invalid_widget_value(c1) or invalid_widget_value(c2) or name == "":
        return

    if c1 == c2:
        print("⚠️ Elige columnas distintas")
        return

    save_snapshot_fe()
    df_fe[name] = df_fe[c1] - df_fe[c2]

    feature_confirmed = False
    df_balance_input = None

    log_fe("Diferencia", f"{c1} | {c2}", name, "manual")
    refresh_fe()
    update_diff_table()

btn_diff.on_click(aplicar_diff)

# ============================================================
# [12] TAB 5 - BINNING
# ============================================================

bin_table = widgets.HTML()
bin_col = widgets.Dropdown(description="Col:")
bin_edges = widgets.Text(description="Rangos:", value="0,7,30,90,180,700")
bin_labels = widgets.Text(description="Valores:", value="last_minute,short,medium,long,very_long")
bin_name = widgets.Text(description="Nombre:", value="lead_time_group")
btn_bin = widgets.Button(description="Aplicar")

def update_bin_table():
    bin_table.value = f"""
<b> SUGERENCIAS DE BINNING</b><br><br>
{get_bin_recommendation_table()}
"""

def aplicar_bin(b):
    global df_fe, feature_confirmed, df_balance_input

    col = bin_col.value
    name = bin_name.value.strip()

    if df_fe is None or invalid_widget_value(col) or name == "":
        return

    save_snapshot_fe()

    try:
        edges = [float(x.strip()) for x in bin_edges.value.split(",") if x.strip() != ""]
        labels = [x.strip() for x in bin_labels.value.split(",") if x.strip() != ""]
        df_fe[name] = pd.cut(
            df_fe[col],
            bins=edges,
            labels=labels,
            include_lowest=True
        )
    except Exception as e:
        snapshots_fe.pop()
        print(f"⚠️ Revisa rangos y valores. Error: {e}")
        return

    feature_confirmed = False
    df_balance_input = None

    log_fe("Binning", col, "manual_ranges", "manual")
    refresh_fe()
    update_bin_table()

btn_bin.on_click(aplicar_bin)

# ============================================================
# [13] TAB 6 - INTERACCIONES
# ============================================================

inter_table = widgets.HTML()
inter_col1 = widgets.Dropdown(description="Col 1:")
inter_col2 = widgets.Dropdown(description="Col 2:")
inter_method = widgets.Dropdown(options=['multiplicación', 'división'], description="Método:")
inter_name = widgets.Text(description="Nombre:")
inter_dynamic_box = widgets.VBox()
btn_inter = widgets.Button(description="Aplicar")

def update_inter_table():
    inter_table.value = f"""
<b> SUGERENCIAS DE INTERACCIONES</b><br><br>
{get_inter_recommendation_table()}
"""

def update_inter_visibility(change=None):
    inter_dynamic_box.children = [inter_col1, inter_col2, inter_method, inter_name]

inter_method.observe(update_inter_visibility, names='value')

def aplicar_inter(b):
    global df_fe, feature_confirmed, df_balance_input

    c1 = inter_col1.value
    c2 = inter_col2.value
    name = inter_name.value.strip()

    if df_fe is None or invalid_widget_value(c1) or invalid_widget_value(c2) or name == "":
        return

    if c1 == c2:
        print(" Elige columnas distintas")
        return

    save_snapshot_fe()

    if inter_method.value == "multiplicación":
        df_fe[name] = df_fe[c1] * df_fe[c2]
    else:
        df_fe[name] = df_fe[c1] / df_fe[c2].replace(0, np.nan)

    feature_confirmed = False
    df_balance_input = None

    log_fe("Interacción", f"{c1} | {c2}", inter_method.value, "manual")
    refresh_fe()
    update_inter_table()
    update_inter_visibility()

btn_inter.on_click(aplicar_inter)

# ============================================================
# [14] TAB 7 - TRANSFORMACIONES
# ============================================================

trans_table = widgets.HTML()
trans_col = widgets.Dropdown(description="Col:")
trans_name = widgets.Text(description="Nombre:")
btn_trans = widgets.Button(description="Aplicar")
trans_dynamic_box = widgets.VBox()

def update_trans_table():
    trans_table.value = f"""
<b> SUGERENCIAS DE TRANSFORMACIONES</b><br><br>
{get_trans_recommendation_table()}
"""

def autofill_trans_name(change=None):
    col = trans_col.value
    if invalid_widget_value(col):
        return

    if col == "adr":
        trans_name.value = "adr_log"
    elif col == "lead_time":
        trans_name.value = "lead_time_log"
    else:
        trans_name.value = f"{col}_log"

def update_trans_visibility(change=None):
    trans_method_auto = widgets.HTML(
        "<b>Método aplicado automáticamente:</b> log(1 + x)"
    )
    trans_dynamic_box.children = [trans_col, trans_method_auto, trans_name]

trans_col.observe(autofill_trans_name, names='value')

def aplicar_trans(b):
    global df_fe, feature_confirmed, df_balance_input

    col = trans_col.value
    name = trans_name.value.strip()

    if df_fe is None or invalid_widget_value(col) or name == "":
        return

    save_snapshot_fe()
    s = df_fe[col].copy()
    s[s < 0] = 0
    df_fe[name] = np.log1p(s)

    feature_confirmed = False
    df_balance_input = None

    log_fe("Transformación", col, "auto_log", "manual")
    refresh_fe()
    update_trans_table()
    update_trans_visibility()

btn_trans.on_click(aplicar_trans)

# ============================================================
# [15] TAB 8 - SELECCIÓN PRELIMINAR
# ============================================================

sel_title = widgets.HTML("<b> SELECCIÓN FINAL</b>")
sel_table = widgets.HTML()
btn_analyze_corr = widgets.Button(description="Analizar correlación", button_style='warning')

sel_mode = widgets.Dropdown(
    options=['manual+correlación', 'solo manual', 'solo correlación'],
    description="Modo:"
)
sel_corr = widgets.FloatSlider(value=0.90, min=0.70, max=0.99, step=0.01, description="Corr:")
sel_drop_prefix = widgets.Text(
    description="Prefijos:",
    value="reserved_room_type_,assigned_room_type_"
)
sel_drop_cols = widgets.Text(
    description="Columnas:",
    value="is_repeated_guest,arrival_date_week_number,stays_in_week_nights,total_price"
)
sel_dynamic_box = widgets.VBox()
btn_sel = widgets.Button(description="Aplicar")

def update_sel_table():
    if not selection_analysis_done:
        sel_table.value = ""
    else:
        sel_table.value = f"""
<b> SUGERENCIAS DE SELECCIÓN</b><br><br>
{get_selection_recommendation_table()}
"""

def update_sel_visibility(change=None):
    if not selection_analysis_done:
        sel_dynamic_box.children = []
        btn_sel.layout.display = 'none'
        return

    btn_sel.layout.display = ''

    if sel_mode.value == 'solo manual':
        sel_dynamic_box.children = [sel_mode, sel_drop_prefix, sel_drop_cols]
    elif sel_mode.value == 'solo correlación':
        sel_dynamic_box.children = [sel_mode, sel_corr]
    else:
        sel_dynamic_box.children = [sel_mode, sel_corr, sel_drop_prefix, sel_drop_cols]

sel_mode.observe(update_sel_visibility, names='value')

def analizar_correlacion(b):
    global selection_analysis_done

    if df_fe is None:
        return

    with out_selection_analysis:
        out_selection_analysis.clear_output()

        num_df = df_fe.select_dtypes(include=np.number).copy()
        num_df = num_df.loc[:, num_df.nunique(dropna=False) > 1]

        if num_df.shape[1] < 2:
            print("⚠️ No hay suficientes variables numéricas útiles para analizar correlación.")
            return

        corr_matrix = num_df.corr()
        corr_matrix = corr_matrix.dropna(axis=0, how='all').dropna(axis=1, how='all')

        if corr_matrix.shape[0] < 2 or corr_matrix.shape[1] < 2:
            print("⚠️ La matriz de correlación quedó demasiado pequeña después de limpiar columnas constantes.")
            return

        plt.figure(figsize=(12, 8))
        sns.heatmap(corr_matrix, cmap='coolwarm')
        plt.title('Matriz de correlación')
        plt.show()

        corr_pairs = (
            corr_matrix.abs()
            .unstack()
            .reset_index()
        )
        corr_pairs.columns = ['var1', 'var2', 'correlation']
        corr_pairs = corr_pairs[corr_pairs['var1'] != corr_pairs['var2']]

        corr_pairs['pair_key'] = corr_pairs.apply(
            lambda row: tuple(sorted([row['var1'], row['var2']])),
            axis=1
        )
        corr_pairs = corr_pairs.drop_duplicates(subset=['pair_key']).drop(columns=['pair_key'])

        high_corr = corr_pairs[corr_pairs['correlation'] > 0.7].sort_values(
            by='correlation', ascending=False
        )

        print("Columnas actuales del dataset:")
        display(pd.DataFrame({"columnas": df_fe.columns}))

        print("\nCorrelaciones altas (> 0.7):")
        display(high_corr.head(20))

    selection_analysis_done = True
    update_sel_table()
    update_sel_visibility()

btn_analyze_corr.on_click(analizar_correlacion)

def aplicar_sel(b):
    global df_fe, feature_confirmed, df_balance_input

    if df_fe is None:
        return

    save_snapshot_fe()

    cols_to_drop_manual = []
    drop_by_prefix = []
    to_drop_corr = []
    const_cols = []

    if sel_mode.value in ['manual+correlación', 'solo manual']:
        cols_to_drop_manual = [x.strip() for x in sel_drop_cols.value.split(",") if x.strip() != ""]
        prefixes = [x.strip() for x in sel_drop_prefix.value.split(",") if x.strip() != ""]
        for pref in prefixes:
            drop_by_prefix += [c for c in df_fe.columns if c.startswith(pref)]

    if sel_mode.value in ['manual+correlación', 'solo correlación']:
        nun = df_fe.nunique(dropna=False)
        const_cols = nun[nun <= 1].index.tolist()

        num_df = df_fe.select_dtypes(include=np.number)
        if num_df.shape[1] > 1:
            corr = num_df.corr().abs()
            upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
            to_drop_corr = [c for c in upper.columns if any(upper[c] > sel_corr.value)]

    cols_to_drop = list(set(cols_to_drop_manual + drop_by_prefix + const_cols + to_drop_corr))
    cols_to_drop = [c for c in cols_to_drop if c in df_fe.columns]

    if cols_to_drop:
        df_fe.drop(columns=cols_to_drop, inplace=True, errors='ignore')

    feature_confirmed = False
    df_balance_input = None

    log_fe("Selección", "columnas", sel_mode.value, "manual")
    refresh_fe()
    update_sel_table()
    update_sel_visibility()

btn_sel.on_click(aplicar_sel)

# ============================================================
# [16] DESHACER
# ============================================================

btn_undo_fe = widgets.Button(description="Deshacer")

def undo_fe(b):
    global df_fe, history_fe, feature_confirmed, df_balance_input, selection_analysis_done

    if len(snapshots_fe) <= 1:
        return

    snapshots_fe.pop()
    df_fe = snapshots_fe[-1].copy()
    history_fe = history_fe.iloc[:-1].copy()

    feature_confirmed = False
    df_balance_input = None
    selection_analysis_done = False

    with out_selection_analysis:
        out_selection_analysis.clear_output()

    refresh_fe()
    render_history_fe()

    update_direct_table()
    update_ratio_table()
    update_temporal_table()
    update_diff_table()
    update_bin_table()
    update_inter_table()
    update_trans_table()
    update_sel_table()
    update_temporal_visibility()
    update_inter_visibility()
    update_trans_visibility()
    update_sel_visibility()
    autofill_trans_name()

btn_undo_fe.on_click(undo_fe)

# ============================================================
# [17] CONFIRMAR PASO A BALANCEO
# ============================================================

btn_confirm_balance = widgets.Button(
    description="Confirmar para Balanceo",
    button_style='success'
)

def confirmar_balanceo(b):
    global df_fe, df_balance_input, feature_confirmed

    with out_confirm_balance:
        out_confirm_balance.clear_output()

        if df_fe is None:
            print("❌ No hay dataset de feature engineering")
            return

        if df_fe.shape[0] == 0 or df_fe.shape[1] == 0:
            print("❌ El dataset está vacío")
            return

        # eliminar filas con nulos automáticamente
        n_antes = len(df_fe)
        df_temp = df_fe.dropna().copy()
        n_despues = len(df_temp)
        filas_eliminadas = n_antes - n_despues

        if df_temp.shape[0] == 0 or df_temp.shape[1] == 0:
            print("❌ Después de eliminar nulos, el dataset quedó vacío")
            return

        obj_cols = df_temp.select_dtypes(include='object').columns.tolist()
        if len(obj_cols) > 0:
            print("❌ Aún hay columnas object:")
            print(obj_cols)
            return

        # actualizar dataset de trabajo y dataset de salida
        df_fe = df_temp.copy()
        df_balance_input = df_temp.copy()
        feature_confirmed = True

        print("✅ Dataset confirmado para Balanceo")
        print(f"Filas originales: {n_antes}")
        print(f"Filas eliminadas por nulos: {filas_eliminadas}")
        print(f"Filas finales: {df_balance_input.shape[0]}")
        print(f"Columnas finales: {df_balance_input.shape[1]}")

btn_confirm_balance.on_click(confirmar_balanceo)

# ============================================================
# [18] UI FINAL
# ============================================================

tabs_fe = widgets.Tab(children=[
    widgets.VBox([
        direct_table,
        direct_method, direct_col1, direct_col2, direct_col3,
        direct_binary_box,
        direct_name, btn_direct
    ]),
    widgets.VBox([
        ratio_table,
        ratio_col1, ratio_col2, ratio_name, btn_ratio
    ]),
    widgets.VBox([
        temporal_table,
        temporal_dynamic_box,
        btn_temporal
    ]),
    widgets.VBox([
        diff_table,
        diff_col1, diff_col2, diff_name, btn_diff
    ]),
    widgets.VBox([
        bin_table,
        bin_col, bin_edges, bin_labels, bin_name, btn_bin
    ]),
    widgets.VBox([
        inter_table,
        inter_dynamic_box,
        btn_inter
    ]),
    widgets.VBox([
        trans_table,
        trans_dynamic_box,
        btn_trans
    ]),
    widgets.VBox([
        sel_title,
        btn_analyze_corr,
        out_selection_analysis,
        sel_table,
        sel_dynamic_box,
        btn_sel
    ]),
])

for i, t in enumerate([
    "Derivadas",
    "Ratios",
    "Temporales",
    "Diferencias",
    "Binning",
    "Interacciones",
    "Transformaciones",
    "Selección"
]):
    tabs_fe.set_title(i, t)

display(widgets.VBox([
    btn_load_fe,
    tabs_fe,
    widgets.HTML("<h3>Historial</h3>"),
    out_hist_fe,
    btn_undo_fe,
    widgets.HTML("<h3>Paso a Balanceo</h3>"),
    btn_confirm_balance,
    out_confirm_balance
]))

## Paso 4: Class balance + Pipeline

In [27]:
# ============================================================
# PROTOTIPO: CLASS BALANCE + PIPELINE EN MEMORIA
# VERSION 2.2
# Todo se queda dentro del notebook
# ============================================================

# ============================================================
# [1] IMPORTS
# ============================================================

import os
import sys
import pandas as pd
import numpy as np
import ipywidgets as widgets

from IPython.display import display
from datetime import datetime
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

sys.path.append(os.path.abspath(".."))

from src.preprocessing import (
    FeatureBuilder,
    get_column_groups,
    build_preprocessor
)

# ============================================================
# [2] VARIABLES GLOBALES
# ============================================================

df_proto = None

X_proto = None
y_proto = None

X_train_proto = None
X_test_proto = None
y_train_proto = None
y_test_proto = None

X_train_bal_proto = None
y_train_bal_proto = None

X_train_fb_proto = None
X_test_fb_proto = None

X_train_t_proto = None
X_test_t_proto = None

feature_builder_proto = None
preprocessor_proto = None

proto_model_input = None
proto_next_steps_input = None
proto_pipeline_confirmed = False

history_proto = pd.DataFrame(columns=[
    "Paso", "Acción", "Detalle", "Hora"
])

snapshots_proto = []

out_status_proto = widgets.Output()
out_history_proto = widgets.Output()
out_split_proto = widgets.Output()
out_balance_proto = widgets.Output()
out_pipeline_proto = widgets.Output()
out_confirm_proto = widgets.Output()

# ============================================================
# [3] UTILIDADES
# ============================================================

def proto_input_available():
    if 'df_class_balance_input' in globals() and isinstance(df_class_balance_input, pd.DataFrame):
        return True, 'df_class_balance_input'
    if 'df_balance_input' in globals() and isinstance(df_balance_input, pd.DataFrame):
        return True, 'df_balance_input'
    return False, None

def add_history(step, action, detail):
    global history_proto
    row = pd.DataFrame({
        "Paso": [step],
        "Acción": [action],
        "Detalle": [detail],
        "Hora": [datetime.now().strftime("%H:%M:%S")]
    })
    history_proto = pd.concat([history_proto, row], ignore_index=True)
    render_history()

def render_history():
    with out_history_proto:
        out_history_proto.clear_output()
        display(history_proto)

def save_snapshot():
    snapshot = {
        "df_proto": df_proto.copy() if isinstance(df_proto, pd.DataFrame) else None,
        "X_proto": X_proto.copy() if isinstance(X_proto, pd.DataFrame) else None,
        "y_proto": y_proto.copy() if isinstance(y_proto, pd.Series) else None,
        "X_train_proto": X_train_proto.copy() if isinstance(X_train_proto, pd.DataFrame) else None,
        "X_test_proto": X_test_proto.copy() if isinstance(X_test_proto, pd.DataFrame) else None,
        "y_train_proto": y_train_proto.copy() if isinstance(y_train_proto, pd.Series) else None,
        "y_test_proto": y_test_proto.copy() if isinstance(y_test_proto, pd.Series) else None,
        "X_train_bal_proto": X_train_bal_proto.copy() if isinstance(X_train_bal_proto, pd.DataFrame) else None,
        "y_train_bal_proto": y_train_bal_proto.copy() if isinstance(y_train_bal_proto, pd.Series) else None,
        "X_train_fb_proto": X_train_fb_proto.copy() if isinstance(X_train_fb_proto, pd.DataFrame) else None,
        "X_test_fb_proto": X_test_fb_proto.copy() if isinstance(X_test_fb_proto, pd.DataFrame) else None,
        "history_proto": history_proto.copy(),
        "proto_pipeline_confirmed": proto_pipeline_confirmed
    }
    snapshots_proto.append(snapshot)

def restore_snapshot(snapshot):
    global df_proto, X_proto, y_proto
    global X_train_proto, X_test_proto, y_train_proto, y_test_proto
    global X_train_bal_proto, y_train_bal_proto
    global X_train_fb_proto, X_test_fb_proto
    global history_proto, proto_pipeline_confirmed

    df_proto = snapshot["df_proto"]
    X_proto = snapshot["X_proto"]
    y_proto = snapshot["y_proto"]

    X_train_proto = snapshot["X_train_proto"]
    X_test_proto = snapshot["X_test_proto"]
    y_train_proto = snapshot["y_train_proto"]
    y_test_proto = snapshot["y_test_proto"]

    X_train_bal_proto = snapshot["X_train_bal_proto"]
    y_train_bal_proto = snapshot["y_train_bal_proto"]

    X_train_fb_proto = snapshot["X_train_fb_proto"]
    X_test_fb_proto = snapshot["X_test_fb_proto"]

    history_proto = snapshot["history_proto"]
    proto_pipeline_confirmed = snapshot["proto_pipeline_confirmed"]

def safe_series(values):
    if isinstance(values, pd.Series):
        return values
    return pd.Series(values)

# ============================================================
# [4] CARGA DESDE FEATURE
# ============================================================

btn_load_proto = widgets.Button(
    description="Cargar data desde Feature",
    button_style="warning"
)

info_load_proto = widgets.HTML()

def load_from_feature(b):
    global df_proto, X_proto, y_proto
    global X_train_proto, X_test_proto, y_train_proto, y_test_proto
    global X_train_bal_proto, y_train_bal_proto
    global X_train_fb_proto, X_test_fb_proto
    global X_train_t_proto, X_test_t_proto
    global feature_builder_proto, preprocessor_proto
    global proto_model_input, proto_next_steps_input, proto_pipeline_confirmed
    global history_proto, snapshots_proto

    ok, source_name = proto_input_available()

    with out_status_proto:
        out_status_proto.clear_output()

        if not ok:
            print("❌ No encuentro la salida de Feature Engineering.")
            print("Necesitas tener disponible df_balance_input o df_class_balance_input.")
            return

        if source_name == 'df_class_balance_input':
            df_proto = df_class_balance_input.copy()
        else:
            df_proto = df_balance_input.copy()

        X_proto = None
        y_proto = None

        X_train_proto = None
        X_test_proto = None
        y_train_proto = None
        y_test_proto = None

        X_train_bal_proto = None
        y_train_bal_proto = None

        X_train_fb_proto = None
        X_test_fb_proto = None

        X_train_t_proto = None
        X_test_t_proto = None

        feature_builder_proto = None
        preprocessor_proto = None

        proto_model_input = None
        proto_next_steps_input = None
        proto_pipeline_confirmed = False

        history_proto = history_proto.iloc[0:0].copy()
        snapshots_proto = []
        save_snapshot()

        target_dropdown.options = df_proto.columns.tolist()
        if "is_canceled" in df_proto.columns:
            target_dropdown.value = "is_canceled"
        else:
            target_dropdown.value = df_proto.columns[0]

        with out_split_proto:
            out_split_proto.clear_output()
        with out_balance_proto:
            out_balance_proto.clear_output()
        with out_pipeline_proto:
            out_pipeline_proto.clear_output()
        with out_confirm_proto:
            out_confirm_proto.clear_output()

        info_load_proto.value = (
            f"<b>Dataset cargado:</b> {source_name} "
            f"| <b>Shape:</b> {df_proto.shape[0]} filas × {df_proto.shape[1]} columnas"
        )

        add_history("Carga", "Cargar dataset", f"Fuente: {source_name} | Shape: {df_proto.shape}")
        print("✅ Dataset cargado correctamente desde Feature Engineering")

btn_load_proto.on_click(load_from_feature)

# ============================================================
# [5] TAB 1 - SPLIT
# ============================================================

target_dropdown = widgets.Dropdown(description="Target:")
test_size_slider = widgets.FloatSlider(
    value=0.2, min=0.1, max=0.4, step=0.05, description="Test size:"
)
random_state_int = widgets.IntText(value=42, description="Random state:")
btn_split_proto = widgets.Button(description="Aplicar train/test split", button_style="info")

def apply_split(b):
    global X_proto, y_proto
    global X_train_proto, X_test_proto, y_train_proto, y_test_proto
    global X_train_bal_proto, y_train_bal_proto
    global X_train_fb_proto, X_test_fb_proto
    global X_train_t_proto, X_test_t_proto
    global proto_pipeline_confirmed, proto_model_input, proto_next_steps_input

    with out_split_proto:
        out_split_proto.clear_output()

        if df_proto is None:
            print("❌ Primero carga la data desde Feature.")
            return

        target_col = target_dropdown.value

        if target_col not in df_proto.columns:
            print("❌ El target seleccionado no existe.")
            return

        save_snapshot()

        X_proto = df_proto.drop(columns=[target_col]).copy()
        y_proto = safe_series(df_proto[target_col].copy())

        X_train_proto, X_test_proto, y_train_proto, y_test_proto = train_test_split(
            X_proto,
            y_proto,
            test_size=test_size_slider.value,
            stratify=y_proto,
            random_state=random_state_int.value
        )

        X_train_bal_proto = None
        y_train_bal_proto = None
        X_train_fb_proto = None
        X_test_fb_proto = None
        X_train_t_proto = None
        X_test_t_proto = None
        proto_pipeline_confirmed = False
        proto_model_input = None
        proto_next_steps_input = None

        print("Split realizado")
        print(f"X_train: {X_train_proto.shape}")
        print(f"X_test : {X_test_proto.shape}")
        print("\nDistribución y_train:")
        print(y_train_proto.value_counts())

        add_history(
            "Split",
            "Train/Test split",
            f"Target={target_col} | test_size={test_size_slider.value} | random_state={random_state_int.value}"
        )

btn_split_proto.on_click(apply_split)

# ============================================================
# [6] TAB 2 - CLASS BALANCE
# ============================================================

balance_mode = widgets.Dropdown(
    options=["SMOTE", "Sin balanceo"],
    value="SMOTE",
    description="Método:"
)

drop_text_cols = widgets.SelectMultiple(
    options=[],
    description="No numéricas:",
    layout=widgets.Layout(width="650px", height="160px")
)

btn_detect_text = widgets.Button(description="Detectar columnas no numéricas", button_style="info")
btn_balance_proto = widgets.Button(description="Aplicar balanceo", button_style="success")

def detect_text_columns(b):
    with out_balance_proto:
        out_balance_proto.clear_output()

        if X_train_proto is None:
            print(" Primero aplica el split.")
            return

        non_numeric_cols = X_train_proto.select_dtypes(exclude=[np.number]).columns.tolist()
        drop_text_cols.options = non_numeric_cols
        drop_text_cols.value = tuple(non_numeric_cols)

        print(" Columnas no numéricas detectadas:")
        print(non_numeric_cols if len(non_numeric_cols) > 0 else "No se detectaron")

btn_detect_text.on_click(detect_text_columns)

def apply_balance(b):
    global X_train_bal_proto, y_train_bal_proto, X_test_proto
    global proto_pipeline_confirmed, proto_model_input, proto_next_steps_input

    with out_balance_proto:
        out_balance_proto.clear_output()

        if X_train_proto is None or y_train_proto is None:
            print(" Primero aplica el split.")
            return

        save_snapshot()

        cols_to_drop = list(drop_text_cols.value)

        X_train_num = X_train_proto.drop(columns=cols_to_drop, errors="ignore").copy()
        X_test_num = X_test_proto.drop(columns=cols_to_drop, errors="ignore").copy()

        non_numeric_after_drop = X_train_num.select_dtypes(exclude=[np.number]).columns.tolist()

        if len(non_numeric_after_drop) > 0:
            print(" Aún quedan columnas no numéricas antes de SMOTE:")
            print(non_numeric_after_drop)
            print("\nVuelve a detectarlas y elimínalas antes de balancear.")
            return

        if balance_mode.value == "Sin balanceo":
            X_train_bal_proto = X_train_num.copy()
            y_train_bal_proto = y_train_proto.copy()
            metodo = "Sin balanceo"
        else:
            smote = SMOTE(random_state=42)
            X_train_bal_proto, y_train_bal_proto = smote.fit_resample(X_train_num, y_train_proto)
            X_train_bal_proto = pd.DataFrame(X_train_bal_proto, columns=X_train_num.columns)
            y_train_bal_proto = safe_series(y_train_bal_proto)
            metodo = "SMOTE"

        X_test_proto = X_test_num.copy()
        proto_pipeline_confirmed = False
        proto_model_input = None
        proto_next_steps_input = None

        print(" Balanceo completado")
        print(f"Método: {metodo}")
        print(f"X_train_bal: {X_train_bal_proto.shape}")
        print(f"X_test     : {X_test_proto.shape}")
        print("\nDistribución y_train original:")
        print(y_train_proto.value_counts())
        print("\nDistribución y_train balanceado:")
        print(y_train_bal_proto.value_counts())

        add_history(
            "Balanceo",
            "Aplicar balanceo",
            f"Método={metodo} | Columnas eliminadas antes de balanceo={cols_to_drop}"
        )

btn_balance_proto.on_click(apply_balance)

# ============================================================
# [7] TAB 3 - PIPELINE
# ============================================================

btn_run_pipeline = widgets.Button(description="Ejecutar pipeline", button_style="info")

def run_pipeline(b):
    global X_train_fb_proto, X_test_fb_proto
    global X_train_t_proto, X_test_t_proto
    global feature_builder_proto, preprocessor_proto
    global proto_pipeline_confirmed, proto_model_input, proto_next_steps_input

    with out_pipeline_proto:
        out_pipeline_proto.clear_output()

        if X_train_bal_proto is None or y_train_bal_proto is None:
            print(" Primero aplica balanceo.")
            return

        save_snapshot()

        feature_builder_proto = FeatureBuilder()

        X_train_fb_proto = feature_builder_proto.fit_transform(X_train_bal_proto)
        X_test_fb_proto = feature_builder_proto.transform(X_test_proto)

        tmp_df = X_train_fb_proto.copy()
        tmp_df["target"] = y_train_bal_proto.values

        num_cols, cat_cols = get_column_groups(tmp_df, "target")

        preprocessor_proto = build_preprocessor(num_cols, cat_cols)

        X_train_t_proto = preprocessor_proto.fit_transform(X_train_fb_proto)
        X_test_t_proto = preprocessor_proto.transform(X_test_fb_proto)

        proto_model_input = {
            "X_train_raw": X_train_bal_proto.copy(),
            "X_test_raw": X_test_proto.copy(),
            "y_train": y_train_bal_proto.copy(),
            "y_test": y_test_proto.copy(),
            "feature_builder": feature_builder_proto,
            "preprocessor": preprocessor_proto,
            "X_train_t": X_train_t_proto,
            "X_test_t": X_test_t_proto
        }

        proto_next_steps_input = {
            "X_train_raw": X_train_bal_proto.copy(),
            "X_test_raw": X_test_proto.copy(),
            "y_train": y_train_bal_proto.copy(),
            "y_test": y_test_proto.copy(),
            "feature_builder": feature_builder_proto,
            "preprocessor": preprocessor_proto,
            "X_train_t": X_train_t_proto,
            "X_test_t": X_test_t_proto
        }

        print(" Pipeline ejecutado correctamente")
        print(f"Train shape transformado: {X_train_t_proto.shape}")
        print(f"Test shape transformado : {X_test_t_proto.shape}")
        print(f"Numéricas   : {len(num_cols)}")
        print(f"Categóricas : {len(cat_cols)}")
        print("\n Variables creadas:")
        print("- proto_model_input")
        print("- proto_next_steps_input")

        add_history(
            "Pipeline",
            "Ejecutar pipeline reproducible",
            f"Train shape={X_train_t_proto.shape} | Test shape={X_test_t_proto.shape}"
        )

        proto_pipeline_confirmed = False

btn_run_pipeline.on_click(run_pipeline)

# ============================================================
# [8] TAB 4 - CONFIRMAR
# ============================================================

btn_confirm_final = widgets.Button(
    description="Confirmar dataset listo para siguientes pasos",
    button_style="success"
)

def confirm_final(b):
    global proto_pipeline_confirmed, proto_next_steps_input

    with out_confirm_proto:
        out_confirm_proto.clear_output()

        if proto_model_input is None:
            print(" Primero ejecuta el pipeline.")
            return

        proto_pipeline_confirmed = True

        proto_next_steps_input = {
            "X_train_raw": proto_model_input["X_train_raw"],
            "X_test_raw": proto_model_input["X_test_raw"],
            "y_train": proto_model_input["y_train"],
            "y_test": proto_model_input["y_test"],
            "feature_builder": proto_model_input["feature_builder"],
            "preprocessor": proto_model_input["preprocessor"],
            "X_train_t": proto_model_input["X_train_t"],
            "X_test_t": proto_model_input["X_test_t"]
        }

        print(" Dataset confirmado y listo para los siguientes pasos")
        print("\nVariables disponibles en proto_next_steps_input:")
        print("- X_train_raw")
        print("- X_test_raw")
        print("- y_train")
        print("- y_test")
        print("- feature_builder")
        print("- preprocessor")
        print("- X_train_t")
        print("- X_test_t")

        add_history(
            "Confirmación",
            "Confirmar salida final",
            "Se confirmó proto_next_steps_input para la etapa de modelos y evaluación"
        )

btn_confirm_final.on_click(confirm_final)

# ============================================================
# [9] DESHACER
# ============================================================

btn_undo_proto = widgets.Button(description="Deshacer última acción")

def undo_proto(b):
    global snapshots_proto

    if len(snapshots_proto) <= 1:
        with out_status_proto:
            out_status_proto.clear_output()
            print(" No hay acciones para deshacer.")
        return

    snapshots_proto.pop()
    restore_snapshot(snapshots_proto[-1])

    with out_status_proto:
        out_status_proto.clear_output()
        print(" Última acción deshecha.")

    render_history()

btn_undo_proto.on_click(undo_proto)

# ============================================================
# [10] UI
# ============================================================

tab_split = widgets.VBox([
    widgets.HTML("<h4>1. Split</h4>"),
    target_dropdown,
    test_size_slider,
    random_state_int,
    btn_split_proto,
    out_split_proto
])

tab_balance = widgets.VBox([
    widgets.HTML("<h4>2. Class Balance</h4>"),
    balance_mode,
    btn_detect_text,
    drop_text_cols,
    btn_balance_proto,
    out_balance_proto
])

tab_pipeline = widgets.VBox([
    widgets.HTML("<h4>3. Pipeline reproducible</h4>"),
    btn_run_pipeline,
    out_pipeline_proto
])

tab_confirm = widgets.VBox([
    widgets.HTML("<h4>4. Confirmación final</h4>"),
    btn_confirm_final,
    out_confirm_proto
])

tabs_proto = widgets.Tab(children=[tab_split, tab_balance, tab_pipeline, tab_confirm])
tabs_proto.set_title(0, "Split")
tabs_proto.set_title(1, "Balanceo")
tabs_proto.set_title(2, "Pipeline")
tabs_proto.set_title(3, "Confirmar")

display(widgets.VBox([
    widgets.HTML("<h3> Prototipo integrado: Class Balance + Pipeline</h3>"),
    btn_load_proto,
    info_load_proto,
    out_status_proto,
    tabs_proto,
    widgets.HTML("<h3>Historial</h3>"),
    out_history_proto,
    btn_undo_proto
]))

## Paso 5: Modelo predictivo Gradient Boosting

En esta sección se utiliza el modelo `baseline_gb.pkl`, seleccionado por presentar el mejor recall para detectar cancelaciones de reservas hoteleras.

In [29]:
# ============================================================
# PROTOTIPO: MEJOR MODELO BASELINE + EVALUACIÓN
# VERSION 1.0
# ============================================================

# ============================================================
# [1] IMPORTS
# ============================================================

import pandas as pd
import numpy as np
import ipywidgets as widgets
import matplotlib.pyplot as plt

from IPython.display import display
from datetime import datetime

from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    recall_score,
    f1_score,
    accuracy_score,
    precision_score,
    roc_auc_score,
    average_precision_score,
    RocCurveDisplay,
    PrecisionRecallDisplay
)
from sklearn.model_selection import StratifiedKFold, cross_validate

# ============================================================
# [2] VARIABLES GLOBALES
# ============================================================

best_model_proto = None
best_model_pipeline_proto = None

X_train_best_proto = None
X_test_best_proto = None
y_train_best_proto = None
y_test_best_proto = None

feature_builder_best_proto = None
preprocessor_best_proto = None

y_pred_best_proto = None
y_proba_best_proto = None

cv_results_best_proto = None
metrics_summary_best_proto = None

proto_best_model_output = None
best_model_confirmed = False

history_best_proto = pd.DataFrame(columns=[
    "Paso", "Acción", "Detalle", "Hora"
])

snapshots_best_proto = []

out_status_best = widgets.Output()
out_history_best = widgets.Output()
out_train_best = widgets.Output()
out_eval_best = widgets.Output()
out_cv_best = widgets.Output()
out_confirm_best = widgets.Output()

# ============================================================
# [3] UTILIDADES
# ============================================================

def best_input_available():
    return (
        'proto_next_steps_input' in globals()
        and isinstance(proto_next_steps_input, dict)
    )

def add_history_best(step, action, detail):
    global history_best_proto
    row = pd.DataFrame({
        "Paso": [step],
        "Acción": [action],
        "Detalle": [detail],
        "Hora": [datetime.now().strftime("%H:%M:%S")]
    })
    history_best_proto = pd.concat([history_best_proto, row], ignore_index=True)
    render_history_best()

def render_history_best():
    with out_history_best:
        out_history_best.clear_output()
        display(history_best_proto)

def save_snapshot_best():
    snapshot = {
        "best_model_proto": best_model_proto,
        "best_model_pipeline_proto": best_model_pipeline_proto,
        "X_train_best_proto": X_train_best_proto.copy() if isinstance(X_train_best_proto, pd.DataFrame) else None,
        "X_test_best_proto": X_test_best_proto.copy() if isinstance(X_test_best_proto, pd.DataFrame) else None,
        "y_train_best_proto": y_train_best_proto.copy() if isinstance(y_train_best_proto, pd.Series) else None,
        "y_test_best_proto": y_test_best_proto.copy() if isinstance(y_test_best_proto, pd.Series) else None,
        "y_pred_best_proto": y_pred_best_proto.copy() if isinstance(y_pred_best_proto, np.ndarray) else None,
        "y_proba_best_proto": y_proba_best_proto.copy() if isinstance(y_proba_best_proto, np.ndarray) else None,
        "cv_results_best_proto": cv_results_best_proto.copy() if isinstance(cv_results_best_proto, pd.DataFrame) else None,
        "metrics_summary_best_proto": metrics_summary_best_proto.copy() if isinstance(metrics_summary_best_proto, pd.DataFrame) else None,
        "history_best_proto": history_best_proto.copy(),
        "best_model_confirmed": best_model_confirmed
    }
    snapshots_best_proto.append(snapshot)

def restore_snapshot_best(snapshot):
    global best_model_proto, best_model_pipeline_proto
    global X_train_best_proto, X_test_best_proto, y_train_best_proto, y_test_best_proto
    global y_pred_best_proto, y_proba_best_proto
    global cv_results_best_proto, metrics_summary_best_proto
    global history_best_proto, best_model_confirmed

    best_model_proto = snapshot["best_model_proto"]
    best_model_pipeline_proto = snapshot["best_model_pipeline_proto"]
    X_train_best_proto = snapshot["X_train_best_proto"]
    X_test_best_proto = snapshot["X_test_best_proto"]
    y_train_best_proto = snapshot["y_train_best_proto"]
    y_test_best_proto = snapshot["y_test_best_proto"]
    y_pred_best_proto = snapshot["y_pred_best_proto"]
    y_proba_best_proto = snapshot["y_proba_best_proto"]
    cv_results_best_proto = snapshot["cv_results_best_proto"]
    metrics_summary_best_proto = snapshot["metrics_summary_best_proto"]
    history_best_proto = snapshot["history_best_proto"]
    best_model_confirmed = snapshot["best_model_confirmed"]

# ============================================================
# [4] CARGA DESDE EL PASO ANTERIOR
# ============================================================

btn_load_best = widgets.Button(
    description="Cargar data desde Pipeline",
    button_style="warning"
)

info_load_best = widgets.HTML()

def load_from_previous_step(b):
    global X_train_best_proto, X_test_best_proto, y_train_best_proto, y_test_best_proto
    global feature_builder_best_proto, preprocessor_best_proto
    global best_model_proto, best_model_pipeline_proto
    global y_pred_best_proto, y_proba_best_proto
    global cv_results_best_proto, metrics_summary_best_proto
    global proto_best_model_output, best_model_confirmed
    global history_best_proto, snapshots_best_proto

    with out_status_best:
        out_status_best.clear_output()

        if not best_input_available():
            print(" No encuentro proto_next_steps_input.")
            print("Primero confirma el dataset listo en el paso anterior.")
            return

        X_train_best_proto = proto_next_steps_input["X_train_raw"].copy()
        X_test_best_proto = proto_next_steps_input["X_test_raw"].copy()
        y_train_best_proto = proto_next_steps_input["y_train"].copy()
        y_test_best_proto = proto_next_steps_input["y_test"].copy()
        feature_builder_best_proto = proto_next_steps_input["feature_builder"]
        preprocessor_best_proto = proto_next_steps_input["preprocessor"]

        best_model_proto = None
        best_model_pipeline_proto = None
        y_pred_best_proto = None
        y_proba_best_proto = None
        cv_results_best_proto = None
        metrics_summary_best_proto = None
        proto_best_model_output = None
        best_model_confirmed = False

        history_best_proto = history_best_proto.iloc[0:0].copy()
        snapshots_best_proto = []
        save_snapshot_best()

        with out_train_best:
            out_train_best.clear_output()
        with out_eval_best:
            out_eval_best.clear_output()
        with out_cv_best:
            out_cv_best.clear_output()
        with out_confirm_best:
            out_confirm_best.clear_output()

        info_load_best.value = (
            f"<b>Train:</b> {X_train_best_proto.shape} | "
            f"<b>Test:</b> {X_test_best_proto.shape}"
        )

        add_history_best(
            "Carga",
            "Cargar salida anterior",
            f"Train={X_train_best_proto.shape} | Test={X_test_best_proto.shape}"
        )
        print("Datos cargados correctamente desde proto_next_steps_input")

btn_load_best.on_click(load_from_previous_step)

# ============================================================
# [5] TAB 1 - ENTRENAMIENTO
# ============================================================

gb_n_estimators = widgets.IntSlider(value=100, min=50, max=300, step=50, description="n_estimators:")
gb_learning_rate = widgets.FloatSlider(value=0.1, min=0.01, max=0.30, step=0.01, description="learn rate:")
gb_max_depth = widgets.IntSlider(value=3, min=1, max=5, step=1, description="max_depth:")
gb_random_state = widgets.IntText(value=42, description="random state:")
btn_train_best = widgets.Button(description="Entrenar Gradient Boosting", button_style="info")

def train_best_model(b):
    global best_model_proto, best_model_pipeline_proto, best_model_confirmed, proto_best_model_output

    with out_train_best:
        out_train_best.clear_output()

        if X_train_best_proto is None or y_train_best_proto is None:
            print("Primero carga los datos desde el paso anterior.")
            return

        save_snapshot_best()

        best_model_proto = GradientBoostingClassifier(
            n_estimators=gb_n_estimators.value,
            learning_rate=gb_learning_rate.value,
            max_depth=gb_max_depth.value,
            random_state=gb_random_state.value
        )

        best_model_pipeline_proto = Pipeline([
            ("feature_builder", feature_builder_best_proto),
            ("preprocessor", preprocessor_best_proto),
            ("clf", best_model_proto)
        ])

        best_model_pipeline_proto.fit(X_train_best_proto, y_train_best_proto)

        best_model_confirmed = False
        proto_best_model_output = None

        print("Modelo entrenado correctamente")
        print("Modelo seleccionado: Gradient Boosting")
        print(f"n_estimators: {gb_n_estimators.value}")
        print(f"learning_rate: {gb_learning_rate.value}")
        print(f"max_depth: {gb_max_depth.value}")

        add_history_best(
            "Entrenamiento",
            "Entrenar mejor modelo",
            f"GB | n_estimators={gb_n_estimators.value} | lr={gb_learning_rate.value} | max_depth={gb_max_depth.value}"
        )

btn_train_best.on_click(train_best_model)

# ============================================================
# [6] TAB 2 - EVALUACIÓN EN TEST
# ============================================================

btn_eval_best = widgets.Button(description="Evaluar en test", button_style="success")

def evaluate_best_model(b):
    global y_pred_best_proto, y_proba_best_proto, metrics_summary_best_proto

    with out_eval_best:
        out_eval_best.clear_output()

        if best_model_pipeline_proto is None:
            print(" Primero entrena el modelo.")
            return

        save_snapshot_best()

        y_pred_best_proto = best_model_pipeline_proto.predict(X_test_best_proto)
        y_proba_best_proto = best_model_pipeline_proto.predict_proba(X_test_best_proto)[:, 1]

        summary = {
            "model": ["Gradient Boosting"],
            "recall": [recall_score(y_test_best_proto, y_pred_best_proto)],
            "precision": [precision_score(y_test_best_proto, y_pred_best_proto)],
            "f1": [f1_score(y_test_best_proto, y_pred_best_proto)],
            "accuracy": [accuracy_score(y_test_best_proto, y_pred_best_proto)],
            "roc_auc": [roc_auc_score(y_test_best_proto, y_proba_best_proto)],
            "avg_precision": [average_precision_score(y_test_best_proto, y_proba_best_proto)]
        }

        metrics_summary_best_proto = pd.DataFrame(summary)

        print("Evaluación completada\n")

        print("Classification Report:")
        print(classification_report(
            y_test_best_proto,
            y_pred_best_proto,
            target_names=["No Cancel", "Cancel"]
        ))

        print("Confusion Matrix:")
        print(confusion_matrix(y_test_best_proto, y_pred_best_proto))

        print("\nResumen de métricas:")
        display(metrics_summary_best_proto)

        plt.figure(figsize=(6, 5))
        RocCurveDisplay.from_estimator(
            best_model_pipeline_proto,
            X_test_best_proto,
            y_test_best_proto,
            name="Gradient Boosting"
        )
        plt.plot([0, 1], [0, 1], "k--")
        plt.title("Curva ROC - Mejor Modelo")
        plt.show()

        plt.figure(figsize=(6, 5))
        PrecisionRecallDisplay.from_estimator(
            best_model_pipeline_proto,
            X_test_best_proto,
            y_test_best_proto,
            name="Gradient Boosting"
        )
        plt.title("Curva Precision-Recall - Mejor Modelo")
        plt.show()

        add_history_best(
            "Evaluación",
            "Evaluar en test",
            f"Recall={metrics_summary_best_proto.loc[0, 'recall']:.4f} | F1={metrics_summary_best_proto.loc[0, 'f1']:.4f}"
        )

btn_eval_best.on_click(evaluate_best_model)

# ============================================================
# [7] TAB 3 - VALIDACIÓN CRUZADA
# ============================================================

cv_folds = widgets.IntSlider(value=5, min=3, max=7, step=1, description="Folds:")
btn_cv_best = widgets.Button(description="Ejecutar cross-validation", button_style="info")

def run_cv_best(b):
    global cv_results_best_proto

    with out_cv_best:
        out_cv_best.clear_output()

        if best_model_pipeline_proto is None:
            print(" Primero entrena el modelo.")
            return

        save_snapshot_best()

        cv = StratifiedKFold(
            n_splits=cv_folds.value,
            shuffle=True,
            random_state=42
        )

        scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

        results = cross_validate(
            best_model_pipeline_proto,
            X_train_best_proto,
            y_train_best_proto,
            cv=cv,
            scoring=scoring,
            n_jobs=-1
        )

        cv_results_best_proto = pd.DataFrame({
            metric: [results[f"test_{metric}"].mean()]
            for metric in scoring
        }, index=["Gradient Boosting"])

        print(" Cross-validation completada")
        display(cv_results_best_proto)

        add_history_best(
            "Cross-validation",
            "Validación cruzada",
            f"Folds={cv_folds.value} | Recall CV={cv_results_best_proto.loc['Gradient Boosting', 'recall']:.4f}"
        )

btn_cv_best.on_click(run_cv_best)

# ============================================================
# [8] TAB 4 - CONFIRMAR
# ============================================================

btn_confirm_best = widgets.Button(
    description="Confirmar mejor modelo para siguientes pasos",
    button_style="success"
)

def confirm_best_model(b):
    global proto_best_model_output, best_model_confirmed

    with out_confirm_best:
        out_confirm_best.clear_output()

        if best_model_pipeline_proto is None or metrics_summary_best_proto is None:
            print(" Primero entrena y evalúa el modelo.")
            return

        proto_best_model_output = {
            "best_model_name": "Gradient Boosting",
            "best_model_pipeline": best_model_pipeline_proto,
            "metrics_test": metrics_summary_best_proto.copy(),
            "metrics_cv": cv_results_best_proto.copy() if isinstance(cv_results_best_proto, pd.DataFrame) else None,
            "X_train": X_train_best_proto.copy(),
            "X_test": X_test_best_proto.copy(),
            "y_train": y_train_best_proto.copy(),
            "y_test": y_test_best_proto.copy(),
            "y_pred_test": y_pred_best_proto.copy() if isinstance(y_pred_best_proto, np.ndarray) else None,
            "y_proba_test": y_proba_best_proto.copy() if isinstance(y_proba_best_proto, np.ndarray) else None
        }

        best_model_confirmed = True

        print(" Mejor modelo confirmado para los siguientes pasos")
        print("Variable creada: proto_best_model_output")
        print("\nContenido principal:")
        print("- best_model_name")
        print("- best_model_pipeline")
        print("- metrics_test")
        print("- metrics_cv")
        print("- y_pred_test")
        print("- y_proba_test")

        add_history_best(
            "Confirmación",
            "Confirmar mejor modelo",
            "Se creó proto_best_model_output"
        )

btn_confirm_best.on_click(confirm_best_model)

# ============================================================
# [9] DESHACER
# ============================================================

btn_undo_best = widgets.Button(description="Deshacer última acción")

def undo_best(b):
    global snapshots_best_proto

    if len(snapshots_best_proto) <= 1:
        with out_status_best:
            out_status_best.clear_output()
            print(" No hay acciones para deshacer.")
        return

        snapshots_best_proto.pop()
    restore_snapshot_best(snapshots_best_proto[-1])

    with out_status_best:
        out_status_best.clear_output()
        print(" Última acción deshecha.")

    render_history_best()

btn_undo_best.on_click(undo_best)

# ============================================================
# [10] UI
# ============================================================

tab_train_best = widgets.VBox([
    widgets.HTML("<h4>1. Entrenamiento del mejor modelo</h4>"),
    gb_n_estimators,
    gb_learning_rate,
    gb_max_depth,
    gb_random_state,
    btn_train_best,
    out_train_best
])

tab_eval_best = widgets.VBox([
    widgets.HTML("<h4>2. Evaluación en test</h4>"),
    btn_eval_best,
    out_eval_best
])

tab_cv_best = widgets.VBox([
    widgets.HTML("<h4>3. Validación cruzada</h4>"),
    cv_folds,
    btn_cv_best,
    out_cv_best
])

tab_confirm_best = widgets.VBox([
    widgets.HTML("<h4>4. Confirmación final</h4>"),
    btn_confirm_best,
    out_confirm_best
])

tabs_best = widgets.Tab(children=[
    tab_train_best,
    tab_eval_best,
    tab_cv_best,
    tab_confirm_best
])

tabs_best.set_title(0, "Entrenar")
tabs_best.set_title(1, "Evaluar")
tabs_best.set_title(2, "Cross-Validation")
tabs_best.set_title(3, "Confirmar")

display(widgets.VBox([
    widgets.HTML("<h3> Prototipo: Mejor modelo baseline + evaluación</h3>"),
    btn_load_best,
    info_load_best,
    out_status_best,
    tabs_best,
    widgets.HTML("<h3>Historial</h3>"),
    out_history_best,
    btn_undo_best
]))